(LaTeX command is defined in this cell.)
$\newcommand{\bm}[1]{\boldsymbol{#1}}$

# Imports

In [1]:
from estimator import *
from estimator.reduction import RC
from sage.all import oo, log

# Lattice Estimator Preliminaries

## Parameter Conversion

### $\mathsf{MLWE}$ to $\mathsf{LWE}$ Parameter Conversion

Let $n,m,q,d \in \mathbb{Z}^+$, $\mathcal{R}=\mathbb{Z}[X]/(X^d+1)$, and $\mathcal{R}_q=\mathbb{Z}_q[X]/(X^d+1)$. Let $\psi$ be a distribution over $\mathbb{Z}$, and by abuse of notation also write $\psi$ for the induced coefficient-wise distribution over $\mathcal{R}_q$ obtained by sampling each coefficient independently from $\psi$ and reducing modulo $q$. Write $\psi^{n+m}$ for the product distribution over $\mathcal{R}_q^{n+m}$. The (decisional) $\mathsf{MLWE}_{n,m,q,d,\psi}$ problem is to distinguish $(\bm{A},\begin{bmatrix}\bm{I}_n \ \bm{A}\end{bmatrix}\bm{s})$ from $(\bm{A},\bm{u})$, where $\bm{A} \stackrel{\$}{\gets} \mathcal{R}_q^{n \times m}$, $\bm{s} \gets \psi^{n + m}$, and $\bm{u} \stackrel{\$}{\gets} \mathcal{R}_q^n$. All arithmetic is performed over $\mathcal{R}_q$. The advantage of an adversary $\mathcal{A}$ against $\mathsf{MLWE}_{n,m,q,d,\psi}$ problem (for security parameter $\kappa$) is defined as $\mathsf{Adv}_\mathcal{A}^\mathsf{MLWE}(\kappa) := \left|\Pr[1\gets \mathcal{A}(\bm{A},\begin{bmatrix}\bm{I}_n \ \bm{A}\end{bmatrix}\bm{s})] - \Pr[1 \gets \mathcal{A}(\bm{A},\bm{u})]\right|$, where the probabilities are taken over the stated sampling and the internal randomness of $\mathcal{A}$.

We recall that $\mathsf{MLWE}_{n,m,q,d,\sigma} \rightsquigarrow \mathsf{LWE}_{md,nd,q,\sigma}$, i.e., the lattice estimator consider the LWE of the form $\mathsf{LWE}_{N_\text{LWE},M_\text{LWE},q,\chi_{\bm{s}},\chi_{\bm{e}}}$.

In [2]:
def MLWE_to_LWE_params(n_rows, m_cols, q, d, sigma, tag=None):
    """
    Convert MLWE_{n,m,q,d,sigma} into coefficient-flattened LWE parameters accepted by lattice estimator.

    Conversion:
        MLWE_{n,m,q,d,sigma} -> LWE_{m*d,n*d,q,sigma}

    Assumptions:
        A in R_q^{n x m}
        b = A*z + e
        z <- D_sigma^m
        e <- D_sigma^n

    Returns:
        LWE.Parameters object.
    """

    N_lwe = m_cols * d      # ordinary LWE secret dimension
    M_lwe = n_rows * d      # ordinary LWE sample count

    psi = ND.DiscreteGaussian(sigma)

    if tag is None:
        tag = (f"MLWE n={n_rows}, m={m_cols}, "f"d={d}, q={q}, sigma={sigma}")

    return LWE.Parameters(
        n=N_lwe,
        q=q,
        Xs=psi,
        Xe=psi,
        m=M_lwe,
        tag=tag
    )

Testing for MLWE to LWE conversion.

In [3]:
n_MLWE = 2; m_MLWE = 4; q_MLWE = 3329; d_MLWE = 512; sigma_MLWE= 3
my_lwe_params = MLWE_to_LWE_params(n_MLWE, m_MLWE, q_MLWE, d_MLWE, sigma_MLWE)
print(my_lwe_params)

LWEParameters(n=2048, q=3329, Xs=D(σ=3.00), Xe=D(σ=3.00), m=1024, tag='MLWE n=2, m=4, d=512, q=3329, sigma=3')


LWE estimate using ADPS16 cost model.

In [4]:
%%time
LWE.estimate(my_lwe_params, red_cost_model = RC.ADPS16)

Algorithm <estimator.lwe_bkw.CodedBKW object at 0x78d4a7052870> on LWEParameters(n=2048, q=3329, Xs=D(σ=3.00), Xe=D(σ=3.00), m=1024, tag='MLWE n=2, m=4, d=512, q=3329, sigma=3') failed with Calling ceil() on infinity or NaN
Algorithm functools.partial(<estimator.lwe_primal.PrimalHybrid object at 0x78d4a7051bb0>, mitm=True, babai=True, red_cost_model=<estimator.reduction.ADPS16 object at 0x78d4a73f2ff0>, red_shape_model=<function GSA at 0x78d4a74022a0>) on LWEParameters(n=2048, q=3329, Xs=D(σ=3.00), Xe=D(σ=3.00), m=1024, tag='MLWE n=2, m=4, d=512, q=3329, sigma=3') failed with The lower bound exceeds the upper bound.
bdd                  :: rop: ≈2^856.1, red: ≈2^511.9, svp: ≈2^856.1, β: 1753, η: ≈2^11.5, d: 3071, tag: bdd
CPU times: user 22.9 s, sys: 74.9 ms, total: 22.9 s
Wall time: 23 s


{'arora-gb': rop: ≈2^inf, dreg: ≈2^inf, tag: arora-gb,
 'usvp': rop: ≈2^inf, tag: usvp,
 'bdd': rop: ≈2^856.1, red: ≈2^511.9, svp: ≈2^856.1, β: 1753, η: ≈2^11.5, d: 3071, tag: bdd,
 'bdd_hybrid': rop: ≈2^856.1, red: ≈2^511.9, svp: ≈2^856.1, β: 1753, η: ≈2^11.5, ζ: 0, |S|: 1, d: 3072, prob: 1.0, ↻: 1, tag: hybrid,
 'dual': rop: ≈2^inf, mem: ≈2^inf, m: ≈2^inf, β: 47, tag: dual,
 'dual_hybrid': rop: ≈2^inf, red: ≈2^inf, guess: ≈2^2782857.0, β: 1023, p: 2, ζ: 0, t: 0, β': 1023, N: ≈2^2782843.4, m: ≈2^11.0}

### $\mathsf{MSIS}$ to $\mathsf{SIS}$ Conversion

Let $n,m,q,d\in\mathbb{Z}^+$, $\mathcal{R}=\mathbb{Z}[X]/(X^d+1)$, and $\mathcal{R}_q=\mathbb{Z}_q[X]/(X^d+1)$. Let $B_\mathsf{MSIS}\in\mathbb{R}^+$. The $\mathsf{MSIS}_{n,m,q,d,B_\mathsf{MSIS}}$ problem is: given $\bm{A}\stackrel{\$}{\gets}\mathcal{R}_q^{n\times m}$, find $\bm{x}\in\mathcal{R}_q^{n+m}$ such that
1. $0<\|\bm{x}\|_2\leq B_\mathsf{MSIS}$, and
2. $\begin{bmatrix}\bm{I}_n\ \bm{A}\end{bmatrix}\bm{x}=\bm{0}$ (over $\mathcal{R}_q$),

where $\|\cdot\|_2$ denotes the Euclidean norm of the coefficient embedding (using canonical representatives) and all arithmetic is performed over $\mathcal{R}_q$. The advantage of a PPT adversary $\mathcal{A}$ against $\mathsf{MSIS}_{n,m,q,d,B_\mathsf{MSIS}}$ (for security parameter $\kappa$) is $\mathsf{Adv}_\mathcal{A}^\mathsf{MSIS}(\kappa):= \Pr\big[\bm{A}\stackrel{\$}{\gets}\mathcal{R}_q^{n \times m}; \bm{x}\gets\mathcal{A}(\bm{A}): \left(0<\|\bm{x}\|_2\leq B_\mathsf{MSIS}\right)\land \left(\begin{bmatrix}\bm{I}_n\ \bm{A}\end{bmatrix}\bm{x}=\bm{0}\right)\big]$, where the probability is over $\bm{A}$ and the internal randomness of $\mathcal{A}$.

We recall that $\mathsf{MSIS}_{n,m,q,d,B_\mathsf{MSIS}} \rightsquigarrow \mathsf{SIS}_{nd,(n+m)d,q,B_\mathsf{MSIS}}$, i.e., the lattice estimator consider SIS of the form $\mathsf{SIS}_{N_\mathsf{MSIS},M_\mathsf{MSIS},q,B_\mathsf{MSIS}}$

In [5]:
def MSIS_to_SIS_params(n_rows, m_random_cols, q, d, B_MSIS, norm=2, tag=None):
    """
    Convert a systematic MSIS instance with matrix [I_n | A], where A is in R_q^{n x m}, into dimensional parameters for the ordinary SIS estimator.

    Dimension mapping:
        coefficient rows    = n*d
        coefficient columns = (n+m)*d
    This counts all coordinates of the systematic MSIS solution, but the lattice estimator does not explicitly model the identity block or the ring/module structure.

    The bound must satisfy B < (q-1)/2.  
    Under balanced representation every vector in Z_q^M has infinity norm at most (q-1)/2, so a bound at or above that makes the instance vacuous; 
    the estimator rejects it rather than returning a cost.  
    This is stricter than B < q, which only rules out the trivial solution q*e_1.

    Returns:
        SIS.Parameters object.
    """
    assert n_rows >= 1 and m_random_cols >= 1 and d >= 1
    assert 0 < B_MSIS < (q - 1)/2, (
        f"need 0 < B < (q-1)/2; the estimator rejects length_bound >= (q-1)/2 "
        f"as trivially easy (B = {B_MSIS}, q = {q})")
    N_sis = n_rows * d
    M_sis = (n_rows + m_random_cols) * d
    if tag is None:
        tag = (f"Systematic MSIS n={n_rows}, m={m_random_cols}, "
               f"d={d}, q={q}, B={B_MSIS}, norm={norm}")
    return SIS.Parameters(n=N_sis, q=q, length_bound=B_MSIS,
                          m=M_sis, norm=norm, tag=tag)

Testing for MSIS to SIS conversion.

In [6]:
n_MSIS = 2; m_MSIS = 4; q_MSIS = 3329; d_MSIS = 512; B_MSIS = 1000
my_sis_params = MSIS_to_SIS_params(n_MSIS,m_MSIS,q_MSIS,d_MSIS,B_MSIS)
print(my_sis_params)

SISParameters(n=1024, q=3329, length_bound=1000, m=3072, norm=2, tag='Systematic MSIS n=2, m=4, d=512, q=3329, B=1000, norm=2')


## Generic Security Analysis

Very roughly, after lattice reduction in dimension $D$, the first vector length is modeled as $\|\mathbf{b}_1\|_2 \approx \delta^{D} \mathrm{vol}(\Lambda)^{1/D}$ where $\mathrm{vol}(\Lambda)$ is the lattice volume/determinant. We have $\delta_0 \approx \left(\frac{\|\mathbf{b}_1\|_2}{\mathrm{vol}(\Lambda)^{1/D}}\right)^{1/D}$.

The value $\delta_0$ depends on the algorithm used:
1. LLL achieves $\delta_0 \approx 1.022$ in polynomial time, but the output quality is weak.
2. BKZ with block size $\beta$ achieves roughly $\delta_0 \approx \left(\frac{\beta}{2\pi e}(\pi \beta)^{1/\beta}\right)^{1/(2(\beta-1))}$.

Typically for:
1. $128$-bit security, we need $\delta_0 \approx 1.0045$,
2. $192$-bit security, we need $\delta_0 \approx 1.0029$,
3. $256$-bit security, we need $\delta_0 \approx 1.0023$.

### Helper function for Bit Security from RHF

In [7]:
from estimator.reduction import RC
from sage.all import oo, log

def log2_cost(x):
    """
    Return the base-2 logarithm of an estimator operation count.
    """
    if x == oo:
        return oo
    return float(log(x, 2))


def RHF_security_label(rhf):
    """
    Assign a heuristic security label based on RHF thresholds.
    """
    rhf = float(rhf)

    if rhf <= 1.0023:
        return "roughly >= 256-bit RHF level"
    elif rhf <= 1.0029:
        return "roughly >= 192-bit RHF level"
    elif rhf <= 1.0045:
        return "roughly >= 128-bit RHF level"
    else:
        return "below 128-bit RHF level"


### Summarizing the Security Estimate

In [8]:
def summarize_security_estimate(results):
    """
    Summarize an LWE or SIS lattice-estimator result.

    Extracts:
        - beta: BKZ block size
        - RHF: root-Hermite factor RC.delta(beta)
        - RHF_level: heuristic RHF security label
        - rop_bits: log2 of total attack cost
        - red_bits: log2 of lattice-reduction cost, if present
        - guess_bits: log2 of guessing cost, if present

    Parameters:
        results:
            Output of one of the following:

                LWE.estimate.rough(params)
                LWE.estimate(params)
                SIS.estimate.rough(params)
                SIS.estimate(params)

            For a single SIS.lattice(params) result, first wrap it as:

                {"lattice": SIS.lattice(params)}

    Returns:
        summary:
            Detailed information for each attack.

        RHF_data:
            BKZ block sizes, root-Hermite factors, and RHF labels.

        cheapest_attack:
            The attack with the smallest finite rop, or None when
            no finite rop is found.
    """
    summary = {}
    RHF_data = {}

    cheapest_attack_name = None
    cheapest_rop_bits = oo

    for attack_name, cost in results.items():
        entry = {}

        # Extract beta and compute the corresponding RHF.
        if "beta" in cost:
            beta = cost["beta"]

            if beta != oo:
                rhf = RC.delta(beta)
                rhf_level = RHF_security_label(rhf)

                entry["beta"] = beta
                entry["RHF"] = rhf
                entry["RHF_level"] = rhf_level

                RHF_data[attack_name] = {
                    "beta": beta,
                    "RHF": rhf,
                    "RHF_level": rhf_level
                }

        # Extract total attack cost.
        if "rop" in cost:
            rop_bits = log2_cost(cost["rop"])
            entry["rop_bits"] = rop_bits

            if rop_bits != oo and rop_bits < cheapest_rop_bits:
                cheapest_rop_bits = rop_bits
                cheapest_attack_name = attack_name

        # Extract lattice-reduction cost, if present.
        if "red" in cost:
            entry["red_bits"] = log2_cost(cost["red"])

        # Extract guessing cost, if present.
        if "guess" in cost:
            entry["guess_bits"] = log2_cost(cost["guess"])

        summary[attack_name] = entry

    if cheapest_attack_name is None:
        cheapest_attack = None
    else:
        cheapest_attack = {
            "attack": cheapest_attack_name,
            "rop_bits": cheapest_rop_bits
        }

    return summary, RHF_data, cheapest_attack

Printing the summary.

In [9]:
def print_security_summary(results, label="Security estimate"):
    """
    Print RHF and concrete bit-security information from an LWE or
    SIS lattice-estimator result.

    Parameters:
        results:
            Output of LWE.estimate.rough(...), LWE.estimate(...),
            SIS.estimate.rough(...), or SIS.estimate(...).

        label:
            Text used in the printed headings.

    Returns:
        summary, RHF_data, cheapest_attack
    """
    summary, RHF_data, cheapest_attack = (
        summarize_security_estimate(results)
    )

    print(f"\nRoot-Hermite factors from {label}:")

    if len(RHF_data) == 0:
        print("No finite beta found, so no RHF was computed.")
    else:
        for attack_name, data in RHF_data.items():
            print(
                f"{attack_name}: "
                f"beta = {data['beta']}, "
                f"RHF = {float(data['RHF'])}, "
                f"{data['RHF_level']}"
            )

    print(f"\nBit-security information from {label}:")

    if cheapest_attack is None:
        print("No finite rop found.")
    else:
        print(
            f"Cheapest finite attack: "
            f"{cheapest_attack['attack']}, "
            f"rop ≈ 2^{cheapest_attack['rop_bits']:.1f}"
        )

    return summary, RHF_data, cheapest_attack

### Some Testing

MLWE testing.

In [10]:
print(my_lwe_params)
lwe_test_result = LWE.estimate.rough(my_lwe_params)
lwe_test_summary, lwe_test_RHF, lwe_test_cheapest = print_security_summary(lwe_test_result,label="LWE.estimate.rough")

LWEParameters(n=2048, q=3329, Xs=D(σ=3.00), Xe=D(σ=3.00), m=1024, tag='MLWE n=2, m=4, d=512, q=3329, sigma=3')

Root-Hermite factors from LWE.estimate.rough:
dual_hybrid: beta = 1023, RHF = 1.002008134629463, roughly >= 256-bit RHF level

Bit-security information from LWE.estimate.rough:
No finite rop found.


MSIS testing.

In [11]:
print(my_sis_params)
sis_test_result = SIS.estimate.rough(my_sis_params)
sis_test_summary, sis_test_RHF, sis_test_cheapest = print_security_summary(sis_test_result,label="SIS.estimate.rough")

SISParameters(n=1024, q=3329, length_bound=1000, m=3072, norm=2, tag='Systematic MSIS n=2, m=4, d=512, q=3329, B=1000, norm=2')
lattice  :: rop: ≈2^460.8, red: ≈2^460.8, δ: 1.001438, β: 1578, d: 2404, tag: euclidean

Root-Hermite factors from SIS.estimate.rough:
lattice: beta = 1578, RHF = 1.0014377568940822, roughly >= 256-bit RHF level

Bit-security information from SIS.estimate.rough:
Cheapest finite attack: lattice, rop ≈ 2^460.8


/home/arzaki/lattice-estimator/estimator/util.py:387: UserWarning: red_shape_model is ignored for Euclidean (norm=2) SIS estimates; reduction shape simulators apply only to infinity-norm attacks.
  y = f(x)


In [12]:
sis_lattice_result = {"lattice": SIS.lattice(my_sis_params)}
sis_lattice_summary, sis_lattice_RHF, sis_lattice_cheapest = print_security_summary(sis_lattice_result,label="SIS.lattice")


Root-Hermite factors from SIS.lattice:
lattice: beta = 1578, RHF = 1.0014377568940822, roughly >= 256-bit RHF level

Bit-security information from SIS.lattice:
Cheapest finite attack: lattice, rop ≈ 2^472.5


## Wrapper Functions for LWE and SIS Analysis

### Wrapper functions for LWE analysis.

In [13]:
### ---- wrapper functions for LWE analysis --------------------------------

def run_LWE_rough_analysis(params, verbose=True):
    """
    Run the rough LWE security estimate and return the results.

    LWE.estimate.rough hard-codes the ADPS16 (core-SVP) cost model together with the ZGSA and costs only usvp and dual_hybrid, so it takes no cost model argument.  
    Because it runs a subset of the attacks, the reported Sec upper-bounds the full ADPS16 estimate.

    Parameters:
        params:
            An LWE.Parameters object, such as the output of MLWE_to_LWE_params(...).
        verbose:
            Print the RHF and bit-security summary.  Set False in sweeps.
    Returns:
        result:
            Raw output of LWE.estimate.rough(params).
        summary:
            Per-attack summary containing beta, RHF, rop_bits, red_bits, and guess_bits when available.
        RHF_data:
            Per-attack BKZ block sizes, RHFs, and RHF labels.
        Sec:
            lg of the smallest finite rop, or None if no attack returned a finite cost.  
            Sec is not reconstructed from beta: bdd and hybrid attacks add an svp or guess term to the reduction cost, so a small beta need not mean a cheap attack.
            None means the instance was not modeled, which is not the same as hard; the caller decides how to treat it.
    """
    result = LWE.estimate.rough(params)
    if verbose:
        summary, RHF_data, cheapest = print_security_summary(result, label="LWE.estimate.rough")
    else:
        summary, RHF_data, cheapest = summarize_security_estimate(result)
    Sec = None if cheapest is None else cheapest['rop_bits']
    return result, summary, RHF_data, Sec


def run_LWE_full_analysis(params, cost_model=RC.ADPS16, verbose=True, deny_list=("arora-gb", "bkw")):
    """
    Run the full LWE security estimate and return the results.

    Costs the whole battery (usvp, bdd, bdd_hybrid, dual, dual_hybrid, and others).  
    Individual algorithms may fail on a given instance; those are reported on stdout by the estimator and are absent from result, 
    so Sec is a minimum over the attacks that succeeded.

    Parameters:
        params:
            An LWE.Parameters object, such as the output of MLWE_to_LWE_params(...).
        cost_model:
            Lattice-reduction cost model.
            RC.ADPS16 is core-SVP; RC.MATZOV is the estimator default and runs roughly 30 bits higher at the block sizes of interest.
        verbose:
            Print the RHF and bit-security summary.  Set False in sweeps.
        deny_list:
            Algorithms excluded from the estimate.  
            BKW and Arora-Gb need far more samples than a module instance provides 
            -- on a 3072-sample instance BKW reports needing 2^798 
            -- so they never enter the minimum, but searching their parameter space costs minutes.
            Excluding them is standard for module-lattice schemes.
            Pass () to cost everything.
    Returns:
        result:
            Raw output of LWE.estimate(params, red_cost_model=cost_model,deny_list=deny_list).
        summary, RHF_data, Sec:
            As in run_LWE_rough_analysis.
    """
    result = LWE.estimate(params, red_cost_model=cost_model,deny_list=deny_list)
    if verbose:
        summary, RHF_data, cheapest = print_security_summary(result, label="LWE.estimate")
    else:
        summary, RHF_data, cheapest = summarize_security_estimate(result)
    Sec = None if cheapest is None else cheapest['rop_bits']
    return result, summary, RHF_data, Sec

Some testing.

In [14]:
%%time
print(my_lwe_params)
lwe_rough_result, lwe_rough_summary, lwe_RHF_rough, lwe_cheapest_rough = run_LWE_rough_analysis(my_lwe_params)

LWEParameters(n=2048, q=3329, Xs=D(σ=3.00), Xe=D(σ=3.00), m=1024, tag='MLWE n=2, m=4, d=512, q=3329, sigma=3')

Root-Hermite factors from LWE.estimate.rough:
dual_hybrid: beta = 1023, RHF = 1.002008134629463, roughly >= 256-bit RHF level

Bit-security information from LWE.estimate.rough:
No finite rop found.
CPU times: user 77.2 ms, sys: 15 μs, total: 77.2 ms
Wall time: 75.8 ms


In [15]:
%%time
print(my_lwe_params)
lwe_full_result, lwe_full_summary, lwe_RHF_full, lwe_cheapest_full = run_LWE_full_analysis(my_lwe_params)

LWEParameters(n=2048, q=3329, Xs=D(σ=3.00), Xe=D(σ=3.00), m=1024, tag='MLWE n=2, m=4, d=512, q=3329, sigma=3')
Algorithm functools.partial(<estimator.lwe_primal.PrimalHybrid object at 0x78d4a7051bb0>, mitm=True, babai=True, red_cost_model=<estimator.reduction.ADPS16 object at 0x78d4a73f2ff0>, red_shape_model=<function GSA at 0x78d4a74022a0>) on LWEParameters(n=2048, q=3329, Xs=D(σ=3.00), Xe=D(σ=3.00), m=1024, tag='MLWE n=2, m=4, d=512, q=3329, sigma=3') failed with The lower bound exceeds the upper bound.
bdd                  :: rop: ≈2^856.1, red: ≈2^511.9, svp: ≈2^856.1, β: 1753, η: ≈2^11.5, d: 3071, tag: bdd

Root-Hermite factors from LWE.estimate:
bdd: beta = 1753, RHF = 1.0013239694459048, roughly >= 256-bit RHF level
bdd_hybrid: beta = 1753, RHF = 1.0013239694459048, roughly >= 256-bit RHF level
dual: beta = 47, RHF = 1.0122323027109652, below 128-bit RHF level
dual_hybrid: beta = 1023, RHF = 1.002008134629463, roughly >= 256-bit RHF level

Bit-security information from LWE.estim

### Wrapper Functions for SIS Analysis

In [16]:
### ---- wrapper functions for SIS analysis --------------------------------

def run_SIS_rough_analysis(params, verbose=True):
    """
    Run the rough SIS security estimate and return the results.

    SIS.estimate.rough hard-codes the ADPS16 (core-SVP) cost model, so it takes no cost model argument.  
    There is no runtime advantage over the full estimate: SIS costs a single lattice attack either way.  
    Retained for symmetry with the LWE wrappers.

    Parameters:
        params:
            A SIS.Parameters object, such as the output of MSIS_to_SIS_params(...).
        verbose:
            Print the RHF and bit-security summary.  Set False in sweeps.
    Returns:
        result, summary, RHF_data, Sec — as in run_LWE_rough_analysis.
    """
    result = SIS.estimate.rough(params)
    if verbose:
        summary, RHF_data, cheapest = print_security_summary(
            result, label="SIS.estimate.rough")
    else:
        summary, RHF_data, cheapest = summarize_security_estimate(result)
    Sec = None if cheapest is None else cheapest['rop_bits']
    return result, summary, RHF_data, Sec


def run_SIS_full_analysis(params, cost_model=RC.ADPS16, verbose=True):
    """
    Run the full SIS security estimate and return the results.

    Parameters:
        params:
            A SIS.Parameters object, such as the output of MSIS_to_SIS_params(...).
        cost_model:
            Lattice-reduction cost model.  RC.ADPS16 is core-SVP; RC.MATZOV is the estimator default.
        verbose:
            Print the RHF and bit-security summary.  Set False in sweeps.
    Returns:
        result, summary, RHF_data, Sec — as in run_LWE_rough_analysis.
    """
    result = SIS.estimate(params, red_cost_model=cost_model)
    if verbose:
        summary, RHF_data, cheapest = print_security_summary(
            result, label="SIS.estimate")
    else:
        summary, RHF_data, cheapest = summarize_security_estimate(result)
    Sec = None if cheapest is None else cheapest['rop_bits']
    return result, summary, RHF_data, Sec

Some testing.

In [17]:
%%time
print(my_sis_params)
sis_rough_result, sis_rough_summary, sis_RHF_rough, sis_cheapest_rough = run_SIS_rough_analysis(my_sis_params)

SISParameters(n=1024, q=3329, length_bound=1000, m=3072, norm=2, tag='Systematic MSIS n=2, m=4, d=512, q=3329, B=1000, norm=2')
lattice  :: rop: ≈2^460.8, red: ≈2^460.8, δ: 1.001438, β: 1578, d: 2404, tag: euclidean

Root-Hermite factors from SIS.estimate.rough:
lattice: beta = 1578, RHF = 1.0014377568940822, roughly >= 256-bit RHF level

Bit-security information from SIS.estimate.rough:
Cheapest finite attack: lattice, rop ≈ 2^460.8
CPU times: user 1.09 ms, sys: 11 μs, total: 1.1 ms
Wall time: 969 μs


In [18]:
%%time
print(my_sis_params)
sis_full_result, sis_full_summary, sis_RHF_full, sis_cheapest_full = run_SIS_full_analysis(my_sis_params)

SISParameters(n=1024, q=3329, length_bound=1000, m=3072, norm=2, tag='Systematic MSIS n=2, m=4, d=512, q=3329, B=1000, norm=2')
lattice  :: rop: ≈2^460.8, red: ≈2^460.8, δ: 1.001438, β: 1578, d: 2404, tag: euclidean

Root-Hermite factors from SIS.estimate:
lattice: beta = 1578, RHF = 1.0014377568940822, roughly >= 256-bit RHF level

Bit-security information from SIS.estimate:
Cheapest finite attack: lattice, rop ≈ 2^460.8
CPU times: user 545 μs, sys: 333 μs, total: 878 μs
Wall time: 819 μs


### Testing for Large Quantities

Testing for:
1. $\mathsf{MLWE}_{p,\ell',q,d,\sigma_\mathrm{MQ}}$,
2. $\mathsf{MLWE}_{p,\ell',q,d,\sigma_\mathrm{AK}}$,
3. $\mathsf{MSIS}_{p,\ell'+1,q,d,B_\mathsf{MSIS}}$,

where $p = 2$, $\ell' = 2$, $q = 2^{64}$, $d = 512$, $\sigma_\mathrm{MQ} = 2^{5.134}$, $\sigma_\mathrm{AK} = 2^{3.006}$.

In [19]:
from sage.all import RR

# Common parameters
p = 2; ell_prime = 2; q = 2^64; d = 512

# MLWE Gaussian standard deviations
sigma_MQ = RR(2)^RR("5.134")
sigma_AK = RR(2)^RR("3.006")

# MSIS Euclidean norm bound
B_MSIS = RR(2)^RR("59.1849")

print("p =", p)
print("ell_prime =", ell_prime)
print("q =", q)
print("d =", d)
print("sigma_MQ =", sigma_MQ)
print("sigma_AK =", sigma_AK)
print("B_MSIS =", B_MSIS)

p = 2
ell_prime = 2
q = 18446744073709551616
d = 512
sigma_MQ = 35.1146220025358
sigma_AK = 8.03334034591178
B_MSIS = 6.55284945157509e17


In [20]:
my_lwe_params_MQ = MLWE_to_LWE_params(p,ell_prime,q,d,sigma_MQ)
print(my_lwe_params_MQ)
my_lwe_params_AK = MLWE_to_LWE_params(p,ell_prime,q,d,sigma_AK)
print(my_lwe_params_AK)
my_sis_params_BMSIS = MSIS_to_SIS_params(p,ell_prime+1,q,d,B_MSIS,norm=2)
print(my_sis_params_BMSIS)

LWEParameters(n=1024, q=18446744073709551616, Xs=D(σ=35.11), Xe=D(σ=35.11), m=1024, tag='MLWE n=2, m=2, d=512, q=18446744073709551616, sigma=35.1146220025358')
LWEParameters(n=1024, q=18446744073709551616, Xs=D(σ=8.03), Xe=D(σ=8.03), m=1024, tag='MLWE n=2, m=2, d=512, q=18446744073709551616, sigma=8.03334034591178')
SISParameters(n=1024, q=18446744073709551616, length_bound=6.55284945157509e17, m=2560, norm=2, tag='Systematic MSIS n=2, m=3, d=512, q=18446744073709551616, B=6.55284945157509e17, norm=2')


In [21]:
%%time
(lwe_MQ_rough_result,lwe_MQ_rough_summary,lwe_MQ_RHF_rough,lwe_MQ_cheapest_rough) = run_LWE_rough_analysis(my_lwe_params_MQ)

usvp                 :: rop: ≈2^31.5, red: ≈2^31.5, δ: 1.008910, β: 108, d: 2047, tag: usvp
dual_hybrid          :: rop: ≈2^33.5, red: ≈2^33.3, guess: ≈2^30.6, β: 114, p: 3, ζ: 0, t: 0, β': 114, N: 1044.748, m: 1024

Root-Hermite factors from LWE.estimate.rough:
usvp: beta = 108, RHF = 1.0089095803238675, below 128-bit RHF level
dual_hybrid: beta = 114, RHF = 1.008665241318469, below 128-bit RHF level

Bit-security information from LWE.estimate.rough:
Cheapest finite attack: usvp, rop ≈ 2^31.5
CPU times: user 307 ms, sys: 3.99 ms, total: 311 ms
Wall time: 311 ms


In [22]:
%%time
(lwe_AK_rough_result,lwe_AK_rough_summary,lwe_AK_RHF_rough,lwe_AK_cheapest_rough) = run_LWE_rough_analysis(my_lwe_params_AK)

usvp                 :: rop: ≈2^27.2, red: ≈2^27.2, δ: 1.009588, β: 93, d: 2025, tag: usvp
dual_hybrid          :: rop: ≈2^28.7, red: ≈2^28.3, guess: ≈2^26.5, β: 97, p: 3, ζ: 0, t: 0, β': 97, N: 1137.413, m: 1024

Root-Hermite factors from LWE.estimate.rough:
usvp: beta = 93, RHF = 1.0095878948759565, below 128-bit RHF level
dual_hybrid: beta = 97, RHF = 1.0093969875477766, below 128-bit RHF level

Bit-security information from LWE.estimate.rough:
Cheapest finite attack: usvp, rop ≈ 2^27.2
CPU times: user 590 ms, sys: 0 ns, total: 590 ms
Wall time: 589 ms


In [23]:
%%time
(sis_full_result,sis_full_summary,sis_RHF_full,sis_cheapest_full) = run_SIS_rough_analysis(my_sis_params_BMSIS)

lattice  :: rop: ≈2^28.9, red: ≈2^28.9, δ: 1.009304, β: 99, d: 2214, tag: euclidean

Root-Hermite factors from SIS.estimate.rough:
lattice: beta = 99, RHF = 1.0093043507947785, below 128-bit RHF level

Bit-security information from SIS.estimate.rough:
Cheapest finite attack: lattice, rop ≈ 2^28.9
CPU times: user 2.32 ms, sys: 0 ns, total: 2.32 ms
Wall time: 2.32 ms


# Input Preparations

$\mathsf{LaTSEC}$ parameters input.

Updated bound for $\beta_\bm{\lambda}$ and $\omega_{\bm{\lambda}}$.

We set $(\beta_{\bm{\lambda}},\omega_{\bm{\lambda}}) = (5,25)$ for $n = 40$ and $(\beta_{\bm{\lambda}},\omega_{\bm{\lambda}}) = (9,45)$ for $n = 60$.

The tight bound is $(\beta_{\bm{\lambda}},\omega_{\bm{\lambda}}) = (4,24)$ for $n = 40$ and $(\beta_{\bm{\lambda}},\omega_{\bm{\lambda}}) = (8,42)$ for $n = 60$.

In [24]:
### ---- real number precision and constants ------------------------------

PREC = RealField(200)
PI   = PREC.pi()          # 200-bit pi, not the symbolic constant
LN2  = PREC(2).log()

### ---- LaTSEC parameter selection: inputs -------------------------------

d = 512

# t_uf := k_p (unforgeability threshold = SEC privacy threshold).
GEOMETRY = [
    dict(n=40, k=20, k_p=5, t_uf=5, alpha_kp=0.464, norm_Ghat=4.432, norm_G=5.861, beta_Q=20.635, c_exp=3, beta_lambda=5,  omega_lambda=25),
    dict(n=60, k=40, k_p=6, t_uf=6, alpha_kp=0.518, norm_Ghat=3.937, norm_G=7.842, beta_Q=26.759, c_exp=3, beta_lambda=9,  omega_lambda=45),
]

# Q_H = 2^63 at kappa=128: Q_H = 2^64 would give Q_H^2 = 2^128 and exhaust
# the T_2 budget on its own. At kappa in {192,256} the extra bit is free.
QUERY = {128: dict(Q_H=2^63, Q_Sign=2^62),
         192: dict(Q_H=2^64, Q_Sign=2^64),
         256: dict(Q_H=2^64, Q_Sign=2^64)}

KAPPAS   = [128, 192, 256]
LIST_ELL = list(range(8, 26))

### ---- model and search configuration -----------------------------------

SVP_CLASSICAL    = 0.292     # core-SVP sieving exponent; screening only
SVP_QUANTUM      = 0.265
DELTA_Q          = 32        # bit-length window above b_q^min
NTT_REQUIRED     = False     # if True, additionally require q = 1 mod 2*d
ESTIMATOR_COMMIT = "FILL"    # git -C lattice-estimator log -1 --format=%H

### ---- input validation --------------------------------------------------

assert d & (d - 1) == 0, "d must be a power of two for R = Z[X]/(X^d+1)"
assert all(g['t_uf'] == g['k_p'] for g in GEOMETRY), "t_uf must equal k_p"
assert all(ell >= 2 for ell in LIST_ELL)
assert set(QUERY) == set(KAPPAS)

for kap in KAPPAS:
    Q_H, Q_S = QUERY[kap]['Q_H'], QUERY[kap]['Q_Sign']
    for g in GEOMETRY:
        # T_1 = (n+1)(Q_H+1) Q_Sign 2^(-d+1) <= 2^-kappa
        assert (g['n'] + 1) * (Q_H + 1) * Q_S <= 2^(d - 1 - kap), \
            f"T_1 fails: kappa={kap}, n={g['n']}"
        # T_2 = (Q_H^2 + t_uf Q_Sign Q_H) 2^(-2 kappa) <= 2^-kappa.
        # Tight: holds with equality at kappa=128 for t_uf = 6.
        assert Q_H^2 + g['t_uf'] * Q_S * Q_H <= 2^kap, \
            f"T_2 fails: kappa={kap}, t_uf={g['t_uf']}"

Derived quantities used for inputs.

In [25]:
### ---- derived quantities ------------------------------------------------

# T_5: |C| = 2^w * binomial(d,w) >= Q_H * 2^kappa
def _omega_c(kap):
    for w in range(1, 400):
        if 2^w * binomial(d, w) >= QUERY[kap]['Q_H'] * 2^kap:
            return w
    assert False, f"no omega_c found for kappa={kap}"

OMEGA_C = {kap: _omega_c(kap) for kap in KAPPAS}

# T_3: half/half split gives eps = 2^-(kappa+2) and
# kappa_MLWE = ceil(kappa + 1 + lg k) = kappa + 1 + ceil(lg k), keyed by (kappa, k).
EPSILON    = {kap: 2^(-(kap + 2)) for kap in KAPPAS}
KAPPA_MLWE = {(kap, g['k']): kap + 1 + ceil(log(g['k'], 2))
              for kap in KAPPAS for g in GEOMETRY}

# Screening block sizes (core-SVP, classical); exact since SVP_* are rationals.
BETA_KAPPA = {kap: ceil(kap / SVP_CLASSICAL) for kap in KAPPAS}
BETA_MLWE  = {key: ceil(val / SVP_CLASSICAL) for key, val in KAPPA_MLWE.items()}

### ---- validation --------------------------------------------------------

assert OMEGA_C == {128: 30, 192: 44, 256: 60}, OMEGA_C
assert KAPPA_MLWE == {(128, 20): 134, (128, 40): 135,
                      (192, 20): 198, (192, 40): 199,
                      (256, 20): 262, (256, 40): 263}, KAPPA_MLWE
assert all(EPSILON[kap] in QQ for kap in KAPPAS), "EPSILON must stay exact"

print(f"{'kappa':>5} {'k':>3} {'omega_c':>8} {'lg eps':>7} "
      f"{'k_MLWE':>7} {'beta_k':>7} {'beta_MLWE':>10}")
for kap in KAPPAS:
    for g in GEOMETRY:
        print(f"{kap:>5} {g['k']:>3} {OMEGA_C[kap]:>8} "
              f"{-(kap+2):>7} "
              f"{KAPPA_MLWE[(kap, g['k'])]:>7} {BETA_KAPPA[kap]:>7} "
              f"{BETA_MLWE[(kap, g['k'])]:>10}")

kappa   k  omega_c  lg eps  k_MLWE  beta_k  beta_MLWE
  128  20       30    -130     134     439        459
  128  40       30    -130     135     439        463
  192  20       44    -194     198     658        679
  192  40       44    -194     199     658        682
  256  20       60    -258     262     877        898
  256  40       60    -258     263     877        901


Shorthand for $\lg(x) = \log_2(x)$.

In [26]:
def lg(x):
    return PREC(x).log(2)

Precision check for real number.

In [27]:
def assert_prec(store, name, fields):
    r"""Every named field in every entry of `store` must live in PREC."""
    for key, entry in store.items():
        bad = {f: entry[f].parent() for f in fields
               if entry[f].parent() is not PREC}
        assert not bad, f"{name}{key}: not at PREC — {bad}"

# Computing the Smoothing Thresholds

Now to set $S_\mathrm{MQ}$ and $S_\mathrm{AK}$, we consider the approximation $\eta_\epsilon(\mathbb{Z}^N) \approx \sqrt{\frac{\ln(2N/\epsilon)}{\pi}}$ for the smoothing parameter $\eta_\epsilon(\mathbb{Z}^N)$. We define $S(N,\epsilon) := \sqrt{2}\eta_\epsilon(\mathbb{Z}^N) \approx \sqrt{\frac{2\ln(2N/\epsilon)}{\pi}}$. For $\epsilon = 2^{-(\kappa+2)}$, we obtain
\begin{align*}
    S(N,2^{-(\kappa+2)}) = \sqrt{\frac{2(\ln (2N) + (\kappa+2)\ln2)}{\pi}}.
\end{align*}
Hence, the approximation for $S_\mathrm{MQ}$ and $S_\mathrm{AK}$ are as follows:
\begin{align*}
    S_\mathrm{MQ}:= \sqrt{2}\eta_\epsilon(\mathbb{Z}^{d\ell}) &\approx \sqrt{\frac{2(\ln(2d\ell)+(\kappa+2)\ln2)}{\pi}}\\
    S_\mathrm{AK}:= \sqrt{2}\eta_\epsilon(\mathbb{Z}^{kd\ell})&\approx \sqrt{\frac{2(\ln(2kd\ell)+(\kappa+2)\ln2)}{\pi}}
\end{align*}

In [28]:
def S(N, kappa):
    r"""sqrt(2) * eta_eps(Z^N) at eps = EPSILON[kappa], s-parametrization.

    Uses eta_eps(Z^N) ~ sqrt(ln(2N/eps)/pi), so
        S(N, eps) := sqrt(2) * eta_eps(Z^N) ~ sqrt(2*ln(2N/eps)/pi).
    2*N/EPSILON[kappa] is an exact integer, so the logarithm is the only
    inexact step and it is taken at PREC.
    """
    assert N >= 1 and kappa in KAPPAS
    return (2 * (PREC(2*N / EPSILON[kappa])).log() / PI).sqrt()

def S_MQ(kappa, d, ell):
    r"""sqrt(2) * eta_eps(Z^(d*ell))."""
    assert d >= 1 and ell >= 1
    return S(d*ell, kappa)

def S_AK(kappa, k, d, ell):
    r"""sqrt(2) * eta_eps(Z^(k*d*ell))."""
    assert k >= 1 and d >= 1 and ell >= 1
    return S(k*d*ell, kappa)

assert S(512*8, 128).parent().precision() == 200

Printing the smoothing thresholds.

In [29]:
### ---- smoothing thresholds over the search grid -------------------------

SMOOTH = {}
for g in GEOMETRY:
    for kappa in KAPPAS:
        for ell in LIST_ELL:
            SMOOTH[(g['n'], kappa, ell)] = dict(
                N_MQ  = d*ell,
                N_AK  = g['k']*d*ell,
                S_MQ  = S_MQ(kappa, d, ell),
                S_AK  = S_AK(kappa, g['k'], d, ell),
            )

assert_prec(SMOOTH,      'SMOOTH',      ('S_MQ', 'S_AK'))

# S is increasing in N, and k >= 1, so S_AK >= S_MQ for every entry.
for key, v in SMOOTH.items():
    assert v['S_AK'] >= v['S_MQ'], key
    assert 1 < v['S_MQ'] < 100, key

print(f"{'n':>3} {'kappa':>6} {'ell':>4} {'S_MQ':>8} {'S_AK':>8} {'ratio':>7}")
for g in GEOMETRY:
    for kappa in KAPPAS:
        for ell in (LIST_ELL[0], LIST_ELL[-1]):
            v = SMOOTH[(g['n'], kappa, ell)]
            print(f"{g['n']:>3} {kappa:>6} {ell:>4} {v['S_MQ']:>8.4f} "f"{v['S_AK']:>8.4f} {v['S_AK']/v['S_MQ']:>7.4f}")

  n  kappa  ell     S_MQ     S_AK   ratio
 40    128    8   7.9437   8.0628  1.0150
 40    128   25   7.9892   8.1077  1.0148
 40    192    8   9.5574   9.6566  1.0104
 40    192   25   9.5952   9.6941  1.0103
 40    256    8  10.9355  11.0223  1.0079
 40    256   25  10.9686  11.0552  1.0079
 60    128    8   7.9437   8.0901  1.0184
 60    128   25   7.9892   8.1348  1.0182
 60    192    8   9.5574   9.6794  1.0128
 60    192   25   9.5952   9.7168  1.0127
 60    256    8  10.9355  11.0423  1.0098
 60    256   25  10.9686  11.0751  1.0097


# Coefficients and Constants in Gaussian Parameter Expression

For the multi-query privacy reduction, we consider the expression
\begin{align*}
    \frac{1}{\sigma_\mathrm{MQ}^2} = 2 \left(\frac{1}{\sigma_\mathbf{s}^2} + \frac{a_\mathrm{MQ}}{\sigma_\mathbf{R}^2} + \frac{b_\mathrm{MQ}}{\sigma_\mathbf{r}^2}\right),
\end{align*}
where $a_\mathrm{MQ} := k_p /\alpha_{k_p}^2$ and $b_\mathrm{MQ} := \omega_c^2 Q_\mathsf{Sign}\left(\sqrt{n} + \sqrt{k_p} \|\mathbf{\hat{G}}\|_2/\alpha_{k_p}\right)^2$. Also, $\sigma_\mathrm{MQ} \geq S_\mathrm{MQ}$. Meanwhile, for the abort-key simulation, we consider the expression
\begin{align*}
    \frac{1}{\sigma_\mathrm{AK}^2} = 2\beta_\mathbf{Q}^2 \left(\max\left\{\frac{1}{\sigma_\mathbf{s}^2},\frac{1}{\sigma_\mathbf{R}^2}\right\} + \frac{b_\mathrm{AK}}{\sigma_\mathbf{r}^2}\right),
\end{align*}
where $b_\mathrm{AK} := \omega_c^2Q_\mathsf{Sign}\|\mathbf{G}\|_2^2$. Also, $\sigma_\mathrm{AK} \geq S_\mathrm{AK}$.

We write $A_\mathrm{MQ}:= 1/(2S_\mathrm{MQ}^2)$ and $A_\mathrm{AK}:=1/(2\beta_\mathbf{Q}^2S_\mathrm{AK}^2)$, so we obtain
\begin{align*}
    \frac{1}{\sigma_\mathbf{s}^2} + \frac{a_\mathrm{MQ}}{\sigma_\mathbf{R}^2} + \frac{b_\mathrm{MQ}}{\sigma_\mathbf{r}^2} &\leq A_\mathrm{MQ}, \\
    \frac{1}{\sigma_\mathbf{s}^2} + \frac{b_\mathrm{AK}}{\sigma_\mathbf{r}^2} &\leq A_\mathrm{AK}, \\
    \frac{1}{\sigma_\mathbf{R}^2} + \frac{b_\mathrm{AK}}{\sigma_\mathbf{r}^2} &\leq A_\mathrm{AK}.
\end{align*}

## $a_\mathrm{MQ}, b_\mathrm{MQ}, b_\mathrm{AK}$

In [30]:
### ---- coefficients in the Gaussian-parameter expressions ----------------
# 1/sigma_MQ^2 = 2 (1/sigma_s^2 + a_MQ/sigma_R^2 + b_MQ/sigma_r^2)
# 1/sigma_AK^2 = 2 beta_Q^2 (max{1/sigma_s^2, 1/sigma_R^2} + b_AK/sigma_r^2)
# All three are independent of ell.

def a_MQ(k_p, alpha_kp):
    r"""a_MQ := k_p / alpha_{k_p}^2."""
    assert k_p >= 1 and PREC(alpha_kp) > 0
    return PREC(k_p) / PREC(alpha_kp)^2

def b_MQ(omega_c, Q_Sign, n, k_p, norm_Ghat, alpha_kp):
    r"""b_MQ := omega_c^2 Q_Sign (sqrt(n) + sqrt(k_p) ||Ghat||_2 / alpha_{k_p})^2."""
    assert omega_c >= 1 and Q_Sign >= 1 and n >= 1 and k_p >= 1
    assert PREC(norm_Ghat) > 0 and PREC(alpha_kp) > 0
    inner = PREC(n).sqrt() + PREC(k_p).sqrt() * PREC(norm_Ghat) / PREC(alpha_kp)
    return (omega_c^2 * Q_Sign) * inner^2

def b_AK(omega_c, Q_Sign, norm_G):
    r"""b_AK := omega_c^2 Q_Sign ||G||_2^2."""
    assert omega_c >= 1 and Q_Sign >= 1 and PREC(norm_G) > 0
    return (omega_c^2 * Q_Sign) * PREC(norm_G)^2

## $A_\mathrm{MQ}$ and $A_\mathrm{AK}$

In [31]:
def A_MQ(kappa, d, ell):
    r"""1/(2 S_MQ^2) — budget for the multi-query constraint."""
    return 1 / (2 * S_MQ(kappa, d, ell)^2)

def A_AK(kappa, k, d, ell, beta_Q):
    r"""1/(2 beta_Q^2 S_AK^2) — budget for the abort-key constraint."""
    assert PREC(beta_Q) > 0
    return 1 / (2 * PREC(beta_Q)^2 * S_AK(kappa, k, d, ell)^2)

## Deriving the Tuple $(a_\mathrm{MQ},b_\mathrm{MQ},b_\mathrm{AK},A_\mathrm{MQ},A_\mathrm{AK})$

In [32]:
def coeffs_constants(kappa, n, k, k_p, d, ell, alpha_kp, norm_Ghat, norm_G,
                     beta_Q, omega_c, Q_Sign):
    r"""Assemble the constraint-side data. Returns (a_MQ, b_MQ, b_AK, A_MQ, A_AK)."""
    return (a_MQ(k_p, alpha_kp),
            b_MQ(omega_c, Q_Sign, n, k_p, norm_Ghat, alpha_kp),
            b_AK(omega_c, Q_Sign, norm_G),
            A_MQ(kappa, d, ell),
            A_AK(kappa, k, d, ell, beta_Q))

def coeffs_constants_from_dicts(G, query, kappa, d, ell):
    r"""coeffs_constants with arguments read from a GEOMETRY entry and QUERY.

    omega_c is read from OMEGA_C rather than recomputed, so the T_5 search
    lives in exactly one place.
    """
    assert kappa in query and kappa in OMEGA_C
    return coeffs_constants(kappa, G['n'], G['k'], G['k_p'], d, ell,
                            G['alpha_kp'], G['norm_Ghat'], G['norm_G'],
                            G['beta_Q'], OMEGA_C[kappa], query[kappa]['Q_Sign'])

# Optimization Problem to Find $\sigma_\mathbf{s}$, $\sigma_\mathbf{R}$, and $\sigma_\mathbf{r}$

## Some Ingredients to Compute $B_1$

We have $B_1:= \omega_c\sqrt{d\ell}(\beta_\mathbf{s}+c_\mathrm{exp}\beta_\mathbf{R})+\beta_\mathbf{r}$, where
1. $\beta_\mathbf{s} = \theta_\mathbf{s} \sigma_\mathbf{s}\sqrt{d\ell}$, where $\theta_\mathbf{s} \geq \min\left\{1+\frac{\kappa\ln 2}{3d\ell}, 1 + \sqrt{\frac{4\kappa\ln2}{d\ell}}\right\}$,
2. $\beta_\mathbf{R} = \theta_\mathbf{R}\sigma_\mathbf{R}$, where $\theta_\mathbf{R}\geq\min\left\{1+\frac{\kappa_\mathbf{R}\ln 2}{3},1+\sqrt{4\kappa_\mathbf{R}\ln 2}\right\}$ and $\kappa_\mathbf{R}:= \kappa+\left\lceil\log_2((k-1)d\ell)\right\rceil$,
3. $\beta_\mathbf{r} = \theta_\mathbf{r}\sigma_\mathbf{r}\sqrt{d\ell}$, where $\theta_{\bm{r}} \geq \min\left\{ 1+\frac{\kappa_{\bm{r}}\ln 2}{3d\ell}, 1+\sqrt{\frac{4\kappa_{\bm{r}}\ln 2}{d\ell}}\right\}$, $\kappa_{\bm{r}} := \kappa+\left\lceil\log_2(nQ_\mathsf{Sign})\right\rceil$, and $Q_\mathsf{Sign}$ is the maximum number of signing queries.

We define $\theta(\kappa,D) = \min\left\{1+\frac{\kappa\ln 2}{3D},1+\sqrt{\frac{4\kappa\ln 2}{D}}\right\}.$

In [33]:
### ---- Gaussian tail factors and response bounds -------------------------
# theta(kappa_eff, N) := min{1 + kappa_eff ln2/(3N), 1 + sqrt(4 kappa_eff ln2/N)}.
# The paper states theta_* >= some bound; we take the minimal admissible value, which minimizes B_1.

def theta_tail(kappa_eff, N):
    r"""theta(kappa_eff, N). N = 1 gives the undivided form used by theta_R."""
    assert kappa_eff > 0 and N >= 1
    L = PREC(kappa_eff) * LN2
    return min(1 + L/(3*N), 1 + (4*L/N).sqrt())

def beta_s(sigma_s, kappa, d, ell):
    r"""beta_s = theta(kappa, d*ell) * sigma_s * sqrt(d*ell)."""
    assert PREC(sigma_s) > 0 and kappa >= 1 and d >= 1 and ell >= 1
    N = d * ell
    return theta_tail(kappa, N) * PREC(sigma_s) * PREC(N).sqrt()

def beta_R(sigma_R, kappa, k, d, ell):
    r"""beta_R = theta_R * sigma_R, with kappa_R = kappa + ceil(lg((k-1) d ell)).

    No sqrt(d*ell) factor: beta_R bounds a single ring element, so theta_R uses the N = 1 form of theta.
    """
    assert PREC(sigma_R) > 0 and kappa >= 1 and k >= 2 and d >= 1 and ell >= 1
    kappa_R = kappa + Integer((k-1)*d*ell - 1).nbits()
    return theta_tail(kappa_R, 1) * PREC(sigma_R)

def beta_r(sigma_r, kappa, n, d, ell, Q_Sign):
    r"""beta_r = theta(kappa_r, d*ell) * sigma_r * sqrt(d*ell), with kappa_r = kappa + ceil(lg(n Q_Sign))."""
    assert PREC(sigma_r) > 0 and kappa >= 1 and n >= 1 and d >= 1 and ell >= 1
    assert Q_Sign >= 1
    kappa_r = kappa + Integer(n*Q_Sign - 1).nbits()
    N = d * ell
    return theta_tail(kappa_r, N) * PREC(sigma_r) * PREC(N).sqrt()

Observe that $B_1 = (\omega_cd\ell\theta_\mathbf{s})\sigma_\mathbf{s} + (\omega_c c_\mathrm{exp}\theta_\mathbf{R}\sqrt{d\ell})\sigma_\mathbf{R}+(\theta_\mathbf{r}\sqrt{d\ell})\sigma_\mathbf{r}$.
By defining $C_\mathbf{s}:= \omega_cd\ell\theta_\mathbf{s}$, $C_\mathbf{R}:=\omega_c c_\mathrm{exp}\theta_\mathbf{R}\sqrt{d\ell}$, and $C_\mathbf{r}:=\theta_\mathbf{r}\sqrt{d\ell}$, we have $B_1 = C_\mathbf{s}\sigma_\mathbf{s}+C_\mathbf{R}\sigma_\mathbf{R} + C_\mathbf{r}\sigma_\mathbf{r}$.

In [34]:
### ---- objective coefficients --------------------------------------------
# B_1 = C_s sigma_s + C_R sigma_R + C_r sigma_r, obtained by expanding B_1 = omega_c sqrt(d ell) (beta_s + c_exp beta_R) + beta_r.

def C_s(kappa, d, ell, omega_c):
    r"""C_s := omega_c * d * ell * theta_s."""
    assert omega_c >= 1
    N = d * ell
    return omega_c * N * theta_tail(kappa, N)

def C_R(kappa, k, d, ell, omega_c, c_exp):
    r"""C_R := omega_c * c_exp * theta_R * sqrt(d * ell)."""
    assert omega_c >= 1 and k >= 2 and PREC(c_exp) > 0
    N = d * ell
    kappa_R = kappa + Integer((k-1)*N - 1).nbits()
    return omega_c * c_exp * theta_tail(kappa_R, 1) * PREC(N).sqrt()

def C_r(kappa, n, d, ell, Q_Sign):
    r"""C_r := theta_r * sqrt(d * ell).  No omega_c factor."""
    assert n >= 1 and Q_Sign >= 1
    N = d * ell
    kappa_r = kappa + Integer(n*Q_Sign - 1).nbits()
    return theta_tail(kappa_r, N) * PREC(N).sqrt()

## Preparing Coefficients and Constants

Suppose $B_1 = (\omega_cd\ell\theta_\mathbf{s})\sigma_\mathbf{s} + (\omega_c c_\mathrm{exp}\theta_\mathbf{R}\sqrt{d\ell})\sigma_\mathbf{R}+(\theta_\mathbf{r}\sqrt{d\ell})\sigma_\mathbf{r}$. By defining $C_\mathbf{s}:= \omega_cd\ell\theta_\mathbf{s}$, $C_\mathbf{R}:=\omega_c c_\mathrm{exp}\theta_\mathbf{R}\sqrt{d\ell}$, and $C_\mathbf{r}:=\theta_\mathbf{r}\sqrt{d\ell}$, we have $B_1 = C_\mathbf{s}\sigma_\mathbf{s}+C_\mathbf{R}\sigma_\mathbf{R} + C_\mathbf{r}\sigma_\mathbf{r}$. Define $x_\mathbf{s}= 1/\sigma_\mathbf{s}^2$, $x_\mathbf{R} = 1/\sigma_\mathbf{R}^2$, and $x_\mathbf{r} = 1/\sigma_\mathbf{r}^2$, hence $\sigma_\mathbf{s} = 1/\sqrt{x_\mathbf{s}}$, $\sigma_\mathbf{R}= 1/\sqrt{x_\mathbf{R}}$, and $\sigma_\mathbf{r}=1/\sqrt{x_\mathbf{r}}$. Thus, we can write $B_1 = \frac{C_\mathbf{s}}{\sqrt{x_\mathbf{s}}} + \frac{C_\mathbf{R}}{\sqrt{x_\mathbf{R}}} + \frac{C_\mathbf{r}}{\sqrt{x_\mathbf{r}}}$.

Our objective is to minimize $B_1(x_\mathbf{s},x_\mathbf{R},x_\mathbf{r}) = \frac{C_\mathbf{s}}{\sqrt{x_\mathbf{s}}} + \frac{C_\mathbf{R}}{\sqrt{x_\mathbf{R}}} + \frac{C_\mathbf{r}}{\sqrt{x_\mathbf{r}}}$ subject to:
\begin{align*}
    & x_\mathbf{s} + & a_\mathrm{MQ}x_\mathbf{R} + & b_\mathrm{MQ}x_\mathbf{r} &\leq A_\mathrm{MQ},\\
    & x_\mathbf{s}   &                           + & b_\mathrm{AK}x_\mathbf{r} &\leq A_\mathrm{AK},\\
    &                &x_\mathbf{R}               + & b_\mathrm{AK}x_\mathbf{r} &\leq A_\mathrm{AK},
\end{align*}
where $A_\mathrm{MQ}:= 1/(2S_\mathrm{MQ}^2)$ and $A_\mathrm{AK}:=1/(2\beta_\mathbf{Q}^2S_\mathrm{AK}^2)$.

In [35]:
### ---- full coefficient set for the B_1 minimization ---------------------

def objective_coeff_constants(kappa, n, k, k_p, d, ell, alpha_kp, norm_Ghat,
                              norm_G, beta_Q, omega_c, Q_Sign, c_exp):
    r"""All eight coefficients for the B_1 minimization at (kappa, profile, ell).

    Returns (a_MQ, b_MQ, b_AK, A_MQ, A_AK, C_s, C_R, C_r).

    Constraints, with x_s = 1/sigma_s^2 etc.:
        x_s + a_MQ x_R + b_MQ x_r <= A_MQ
        x_s              + b_AK x_r <= A_AK
              x_R        + b_AK x_r <= A_AK
    Objective:
        B_1 = C_s sigma_s + C_R sigma_R + C_r sigma_r
            = C_s/sqrt(x_s) + C_R/sqrt(x_R) + C_r/sqrt(x_r)
    """
    return coeffs_constants(kappa, n, k, k_p, d, ell, alpha_kp, norm_Ghat,
                            norm_G, beta_Q, omega_c, Q_Sign) + (C_s(kappa, d, ell, omega_c),C_R(kappa, k, d, ell, omega_c, c_exp),C_r(kappa, n, d, ell, Q_Sign))

def objective_coeff_constants_from_dicts(G, query, kappa, d, ell):
    r"""objective_coeff_constants with arguments read from GEOMETRY and QUERY."""
    assert kappa in query and kappa in OMEGA_C
    return objective_coeff_constants(
        kappa, G['n'], G['k'], G['k_p'], d, ell, G['alpha_kp'], G['norm_Ghat'], G['norm_G'], G['beta_Q'],
        OMEGA_C[kappa], query[kappa]['Q_Sign'], G['c_exp'])

Generating the table.

In [36]:
### ---- assemble over the search grid -------------------------------------

COEFF = {}
for g in GEOMETRY:
    for kappa in KAPPAS:
        for ell in LIST_ELL:
            (a_mq, b_mq, b_ak, A_mq, A_ak,
             c_s, c_R, c_r) = objective_coeff_constants_from_dicts(
                                  g, QUERY, kappa, d, ell)
            COEFF[(g['n'], kappa, ell)] = dict(
                a_MQ=a_mq, b_MQ=b_mq, b_AK=b_ak, A_MQ=A_mq, A_AK=A_ak,
                C_s=c_s, C_R=c_R, C_r=c_r,
                x_r_max=min(A_mq/b_mq, A_ak/b_ak),
                binding=('AK' if A_ak/b_ak <= A_mq/b_mq else 'MQ'),
            )

FIELDS = ('a_MQ', 'b_MQ', 'b_AK', 'A_MQ', 'A_AK', 'C_s', 'C_R', 'C_r')

assert_prec(COEFF,       'COEFF',       FIELDS + ('x_r_max',))

for key, c in COEFF.items():
    assert c['x_r_max'] > 0, key
    assert all(c[f] > 0 for f in FIELDS), key

def show_coeffs(L=None, fields=FIELDS):
    r"""Print the COEFF entries for ell in L (default: all of LIST_ELL)."""
    L = list(LIST_ELL) if L is None else list(L)
    assert all(ell in LIST_ELL for ell in L), \
        f"not in LIST_ELL: {[ell for ell in L if ell not in LIST_ELL]}"
    print(f"{'n':>3} {'kap':>4} {'ell':>4} | "
          + " ".join(f"lg {f:>6}" for f in fields)
          + f" | {'lg x_r_max':>11} {'bind':>5}")
    for g in GEOMETRY:
        for kappa in KAPPAS:
            for ell in L:
                c = COEFF[(g['n'], kappa, ell)]
                print(f"{g['n']:>3} {kappa:>4} {ell:>4} | "
                      + " ".join(f"{lg(c[f]):>9.4f}" for f in fields)
                      + f" | {lg(c['x_r_max']):>11.4f} {c['binding']:>5}")

Testing

In [37]:
show_coeffs([8,12,16,20,24])

  n  kap  ell | lg   a_MQ lg   b_MQ lg   b_AK lg   A_MQ lg   A_AK lg    C_s lg    C_R lg    C_r |  lg x_r_max  bind
 40  128    8 |    4.5375   81.3956   76.9161   -6.9796  -15.7566   16.9173   16.8876    6.0159 |    -92.6727    AK
 40  128   12 |    4.5375   81.3956   76.9161   -6.9855  -15.7623   17.4988   17.1801    6.3031 |    -92.6784    AK
 40  128   16 |    4.5375   81.3956   76.9161   -6.9897  -15.7664   17.9121   17.3924    6.5080 |    -92.6824    AK
 40  128   20 |    4.5375   81.3956   76.9161   -6.9929  -15.7695   18.2330   17.5533    6.6673 |    -92.6856    AK
 40  128   24 |    4.5375   81.3956   76.9161   -6.9955  -15.7720   18.4953   17.6848    6.7978 |    -92.6881    AK
 40  192    8 |    4.5375   84.5007   80.0212   -7.5132  -16.2771   17.4750   17.6924    6.0212 |    -96.2982    AK
 40  192   12 |    4.5375   84.5007   80.0212   -7.5173  -16.2811   18.0548   17.9849    6.3066 |    -96.3022    AK
 40  192   16 |    4.5375   84.5007   80.0212   -7.5202  -16.2839   18.4

## Optimization Model

We want to find $x,y,z > 0$ such that $G(x,y,z) = \frac{C_x}{\sqrt{x}} + \frac{C_y}{\sqrt{y}} + \frac{C_z}{\sqrt{z}}$ is minimum subject to
\begin{align*}
    x + ay + bz \leq P
    x + cz \leq Q
    y + cz \leq Q
\end{align*}
where $C_x,C_y,C_z,a,b,c \in \mathbb{R}^+$.

We can solve the problem with a one-variable representation. We first fix $z = z_0$ where $0 < z_0 \leq \min\{P/b,Q/c\}$ and so the constraint becomes
\begin{align*}
    x + ay   &\leq P-bz_0, \\
    x        &\leq Q-cz_0, \\
    y        &\leq Q-cz_0.
\end{align*}
The problem becomes minimizing $H(x,y) := \frac{C_x}{\sqrt{x}} + \frac{C_y}{\sqrt{y}}$ subject to $x+ay \leq P_0$, $x \leq Q_0$, and $y \leq Q_0$, where $P_0 = P-bz_0$ and $Q_0 = Q-cz_0$. Using Lagrange multiplier method, we write $\frac{x}{y} = r$ where $r = \left(\frac{aC_x}{C_y}\right)^{2/3}$. We have $x = yr$, and using $x+ay = P_0$, we get the candidate solution $y^{(0)} = \frac{P_0}{a+r}$ and $x^{(0)} = \frac{rP_0}{a+r}$.

Minimizing the function $G(x,y,z)$ with the aforementioned constraint can be transformed into minimizing
\begin{align*}
    \Phi(z):= \frac{C_x}{\sqrt{x^*(z)}} + \frac{C_y}{\sqrt{y^*(z)}} + \frac{C_z}{\sqrt{z}},
\end{align*}
with $0 < z < \min\{P/b,Q/c\}$, where $(x^*(z),y^*(z))$ is a function defined as:
\begin{align*}
    ((x^*(z),y^*(z)) = 
    \begin{cases}
        \left(\frac{rP_0}{a+r},\frac{P_0}{a+r}\right), \text{ if } P_0 \leq (a+r)Q_0 \land P_0 \leq \left(\frac{a+r}{r}\right)Q_0, \\
        \left(Q_0,\frac{P_0-Q_0}{a}\right), \text{ if } \left(\frac{a+r}{r}\right)Q_0 < P_0 \leq (1+a) Q_0, \\ 
        \left(P_0-aQ_0,Q_0\right), \text{ if } (a+r)Q_0 < P_0 \leq (1+a)Q_0, \\
        \left(Q_0,Q_0\right), \text{ if } P_0 > (1+a)Q_0.
    \end{cases}
\end{align*}

In [38]:
### ---- constrained minimization of B_1 -----------------------------------
# Minimize G(x,y,z) = Cx/sqrt(x) + Cy/sqrt(y) + Cz/sqrt(z) subject to x + a y + b z <= P,   x + c z <= Q,   y + c z <= Q,   x,y,z > 0.
# LaTSEC mapping: (x,y,z) = (x_s, x_R, x_r), (Cx,Cy,Cz) = (C_s, C_R, C_r), (a,b,c) = (a_MQ, b_MQ, b_AK), (P,Q) = (A_MQ, A_AK).

_INVPHI = (PREC(5).sqrt() - 1) / 2

def _golden(f, lo, hi, iters=250):
    r"""Golden-section minimum of a unimodal f on [lo, hi], at PREC.
    250 iterations shrink the bracket by ~0.618^250 = 2^-173."""
    a, b = PREC(lo), PREC(hi)
    c, e = b - _INVPHI*(b - a), a + _INVPHI*(b - a)
    fc, fe = f(c), f(e)
    for _ in range(iters):
        if fc < fe:
            b, e, fe = e, c, fc
            c = b - _INVPHI*(b - a); fc = f(c)
        else:
            a, c, fc = c, e, fe
            e = a + _INVPHI*(b - a); fe = f(e)
    return (a + b) / 2

def _two_term(A, B, kap, M):
    r"""argmin A/sqrt(u) + B/sqrt(w) s.t. u + kap*w = M.  Returns w.
    Stationarity gives u/w = (kap*A/B)^(2/3)."""
    rho = (kap * A / B)^(PREC(2)/3)
    return M / (rho + kap)

def _inner(P0, Q0, Cx, Cy, a, r):
    r"""Inner problem at fixed z.  Returns (x, y, case) numbered as in the
    writeup: 1 interior, 2 clamp x, 3 clamp y, 4 both caps."""
    if P0 > (1 + a)*Q0:          return Q0, Q0, 4
    x0, y0 = r*P0/(a + r), P0/(a + r)
    if x0 <= Q0 and y0 <= Q0:    return x0, y0, 1
    if x0 > Q0:                  return Q0, (P0 - Q0)/a, 2
    return P0 - a*Q0, Q0, 3

def minimize_Gxyz_fast(Cx, Cy, Cz, a, b, c, P, Q, verbose=False):
    r"""Minimize G(x,y,z) subject to the three constraints above.

    Partial minimization over (x,y) reduces the problem to a 1-D convex problem in z.  
    Cases 1 and 4 are closed form; cases 2 and 3 are transcendental and use golden section.

    Inputs are non-dimensionalized internally (x/Q, z/Z with Z = min(P/b, Q/c)), so tolerances are relative and the routine works at any scale.  
    Returned x, y, z are in the original units.
    """
    Cx, Cy, Cz = PREC(Cx), PREC(Cy), PREC(Cz)
    a, b, c    = PREC(a), PREC(b), PREC(c)
    P, Q       = PREC(P), PREC(Q)
    assert min(Cx, Cy, Cz, a, b, c, P, Q) > 0, "all parameters must be positive"

    Po, Qo = P, Q                              # originals, for mapping back
    Z = min(P/b, Q/c)
    assert Z > 0, f"empty feasible region: min(P/b, Q/c) = {Z}"
    sQ, sZ = Q.sqrt(), Z.sqrt()
    Cx, Cy, Cz = Cx/sQ, Cy/sQ, Cz/sZ           # objective value is invariant
    b, c       = b*Z/Q, c*Z/Q
    P, Q       = P/Q, PREC(1)

    r    = (a*Cx/Cy)^(PREC(2)/3)
    zmax = PREC(1)                             # exact, by construction of Z
    def xy(z):  return _inner(P - b*z, Q - c*z, Cx, Cy, a, r)
    def Phi(z):
        x, y, _ = xy(z)
        return Cx/x.sqrt() + Cy/y.sqrt() + Cz/z.sqrt()

    TOL  = PREC(2)^(-150)                      # relative; problem is O(1) here
    EDGE = PREC(2)^(-120)
    bps = sorted(set(t for th in (a + r, (a + r)/r, 1 + a)
                     if abs(th*c - b) > TOL*max(th*c, b)
                     for t in [(th*Q - P)/(th*c - b)]
                     if EDGE < t < zmax - EDGE))
    edges = [EDGE] + bps + [zmax - EDGE]

    best_z, best_v = None, None
    for lo, hi in zip(edges[:-1], edges[1:]):
        if hi - lo <= 2*EDGE:  continue
        case = xy((lo + hi)/2)[2]
        if case == 1:
            C1 = (a + r).sqrt() * (Cx/r.sqrt() + Cy)
            z  = _two_term(C1, Cz, b, P)
        elif case == 4:
            z  = _two_term(Cx + Cy, Cz, c, Q)
        else:
            z  = _golden(Phi, lo, hi)
        if not (lo <= z <= hi):                # stationary point outside piece
            z = lo if Phi(lo) < Phi(hi) else hi
        v = Phi(z)
        if best_v is None or v < best_v:  best_v, best_z = v, z

    for t in bps:                              # Phi is kinked here
        v = Phi(t)
        if v < best_v:  best_v, best_z = v, t
    assert best_z is not None, \
        f"no admissible piece found (zmax={zmax}, breakpoints={bps})"

    x, y, case = xy(best_z)
    sl    = (P - (x + a*y + b*best_z), Q - (x + c*best_z), Q - (y + c*best_z))
    scale = PREC(2)^(-100) * max(PREC(1), P, Q)

    if verbose:
        print(f"r = {r}   Z = {Z}   (scaled zmax = 1)")
        print("breakpoints (original units):", [t*Z for t in bps])
        print(f"x={x*Qo}  y={y*Qo}  z={best_z*Z}  G={best_v}  (case {case})")

    return {'x': x*Qo, 'y': y*Qo, 'z': best_z*Z, 'value': best_v,
            'case': case, 'r': r, 'breakpoints': [t*Z for t in bps],
            'active': [nm for nm, s in zip(('budget','cap_x','cap_y'), sl)
                       if abs(s) <= scale],
            'feasible': min(sl) >= -scale,
            'at_kink': any(abs(best_z - t) <= TOL*max(PREC(1), t) for t in bps)}

## Solving $\min_{x_\mathbf{s},x_\mathbf{R},x_\mathbf{r}} B_1(x_\mathbf{s},x_\mathbf{R},x_\mathbf{r})$

Converting $x$-values to $\sigma$-values. Since $x = \frac{1}{\sigma^2}$, then $\sigma = \frac{1}{\sqrt{x}}$.

In [39]:
def x_to_sigma(x):
    r"""sigma = 1/sqrt(x), inverting x = 1/sigma^2."""
    assert PREC(x) > 0
    return 1 / PREC(x).sqrt()

def solution_to_sigmas(sol):
    r"""(sigma_s, sigma_R, sigma_r) from a minimize_Gxyz_fast result.

    The solver's (x, y, z) are (x_s, x_R, x_r) under the LaTSEC mapping
    (Cx,Cy,Cz) = (C_s,C_R,C_r), (a,b,c) = (a_MQ,b_MQ,b_AK), (P,Q) = (A_MQ,A_AK).
    """
    assert sol['feasible'], f"infeasible solution (case {sol['case']})"
    return tuple(x_to_sigma(sol[key]) for key in ('x', 'y', 'z'))

Minimizing $B_1$ given $C = (C_\mathbf{s},C_\mathbf{R},C_\mathbf{r})$ and $const\_coeff = (a_\mathrm{MQ},b_\mathrm{MQ},b_\mathrm{AK},A_\mathrm{MQ},A_\mathrm{AK})$

In [40]:
### ---- step 8: jointly optimize the Gaussian widths ----------------------

def minimize_B1_from_tuples(C, const_coeff):
    r"""Minimize B_1 given C = (C_s, C_R, C_r) and const_coeff = (a_MQ, b_MQ, b_AK, A_MQ, A_AK).

    Returns the optimized widths and B_1 only; the source widths sigma_MQ, sigma_AK and their smoothing ratios are in the next step.
    """
    C_s, C_R, C_r = C
    a_MQ, b_MQ, b_AK, A_MQ, A_AK = const_coeff
    sol = minimize_Gxyz_fast(C_s, C_R, C_r, a_MQ, b_MQ, b_AK, A_MQ, A_AK)
    assert sol['feasible'], f"infeasible solution (case {sol['case']})"
    return dict(
        sigma_s=x_to_sigma(sol['x']),
        sigma_R=x_to_sigma(sol['y']),
        sigma_r=x_to_sigma(sol['z']),
        x_s=sol['x'], x_R=sol['y'], x_r=sol['z'],
        B_1=sol['value'], case=sol['case'], active=sol['active'],
    )


def minimize_B1_from_dicts(key):
    r"""minimize_B1_from_tuples with arguments read from COEFF."""
    c = COEFF[key]
    return minimize_B1_from_tuples(
        (c['C_s'], c['C_R'], c['C_r']),
        (c['a_MQ'], c['b_MQ'], c['b_AK'], c['A_MQ'], c['A_AK']))


WIDTHS = {key: minimize_B1_from_dicts(key) for key in COEFF}

# MQ inactive on the current profiles, so the inner problem lands on case 4.
for key, w in WIDTHS.items():
    assert w['case'] == 4, f"{key}: MQ constraint active (case {w['case']})"

Constructing the table for diagnostic.

In [41]:
### ---- solve step 8 over the search grid ---------------------------------

WIDTHS = {key: minimize_B1_from_dicts(key) for key in COEFF}

# MQ inactive on the current profiles, so the inner problem lands on case 4.
for key, w in WIDTHS.items():
    assert w['case'] == 4, f"{key}: MQ constraint active (case {w['case']})"

assert_prec(WIDTHS,      'WIDTHS',      ('sigma_s', 'sigma_R', 'sigma_r','x_s', 'x_R', 'x_r', 'B_1'))

def minimize_B1_table(L=None):
    r"""Print the optimized widths for ell in L (default: all of LIST_ELL).

    Step 8 only: sigma_MQ, sigma_AK and the ratios R_MQ, R_AK are Step 9.
    """
    L = list(LIST_ELL) if L is None else list(L)
    assert all(ell in LIST_ELL for ell in L), \
        f"not in LIST_ELL: {[ell for ell in L if ell not in LIST_ELL]}"

    print(f"{'n':>3} {'kap':>4} {'ell':>4} | {'lg sig_s':>9} {'lg sig_R':>9} "
          f"{'lg sig_r':>9} | {'lg B_1':>9}")
    for g in GEOMETRY:
        for kappa in KAPPAS:
            for ell in L:
                w = WIDTHS[(g['n'], kappa, ell)]
                print(f"{g['n']:>3} {kappa:>4} {ell:>4} | "
                      f"{lg(w['sigma_s']):>9.4f} {lg(w['sigma_R']):>9.4f} "
                      f"{lg(w['sigma_r']):>9.4f} | {lg(w['B_1']):>9.4f}")

Some testing.

In [42]:
minimize_B1_table([8,12,16,20,24])

  n  kap  ell |  lg sig_s  lg sig_R  lg sig_r |    lg B_1
 40  128    8 |   16.7354   16.7354   46.3363 |   52.3522
 40  128   12 |   16.6855   16.6855   46.3392 |   52.6423
 40  128   16 |   16.6467   16.6467   46.3412 |   52.8492
 40  128   20 |   16.6156   16.6156   46.3428 |   53.0101
 40  128   24 |   16.5892   16.5892   46.3441 |   53.1419
 40  192    8 |   17.2865   17.2865   48.1491 |   54.1703
 40  192   12 |   17.2395   17.2395   48.1511 |   54.4577
 40  192   16 |   17.2031   17.2031   48.1525 |   54.6632
 40  192   20 |   17.1737   17.1737   48.1536 |   54.8231
 40  192   24 |   17.1486   17.1486   48.1545 |   54.9541
 40  256    8 |   17.4441   17.4441   48.7874 |   54.8137
 40  256   12 |   17.3993   17.3993   48.7889 |   55.0990
 40  256   16 |   17.3647   17.3647   48.7900 |   55.3032
 40  256   20 |   17.3366   17.3366   48.7909 |   55.4624
 40  256   24 |   17.3126   17.3126   48.7916 |   55.5929
 60  128    8 |   17.2545   17.2545   47.1362 |   53.1521
 60  128   12 

## Deriving $\sigma_\mathrm{MQ}$ and $\sigma_\mathrm{AK}$ and Some Checking

Now, we compute the following quantities of $\sigma_\mathrm{MQ}$ and $\sigma_\mathrm{AK}$ defined as follows:
\begin{align*}
    \frac{1}{\sigma_\mathrm{MQ}^2} &:= 2 \left(\frac{1}{\sigma_\mathbf{s}^2} + \frac{a_\mathrm{MQ}}{\sigma_\mathbf{R}^2} + \frac{b_\mathrm{MQ}}{\sigma_\mathbf{r}^2}\right),\\
    \frac{1}{\sigma_\mathrm{AK}^2} &:= 2 \beta_\mathbf{Q}^2\left(\max\left\{\frac{1}{\sigma_\mathbf{s}^2},\frac{1}{\sigma_\mathbf{R}^2}\right\} + \frac{b_\mathrm{AK}}{\sigma_\mathbf{r}^2}\right),
\end{align*}
that is
\begin{align*}
    \sigma_\mathrm{MQ} &:= \frac{1}{\sqrt{2\left(\frac{1}{\sigma_\mathbf{s}^2} + \frac{a_\mathrm{MQ}}{\sigma_\mathbf{R}^2} + \frac{b_\mathrm{MQ}}{\sigma_\mathbf{r}^2}\right)}},\\
    \sigma_\mathrm{AK} &:= \frac{1}{\beta_\mathbf{Q}\sqrt{2\left(\max\left\{\frac{1}{\sigma_\mathbf{s}^2},\frac{1}{\sigma_\mathbf{R}^2}\right\} + \frac{b_\mathrm{AK}}{\sigma_\mathbf{r}^2}\right)}}
\end{align*}
We also want to compute $R_\mathrm{MQ}$ and $R_\mathrm{AK}$ defined as:
\begin{align*}
    R_\mathrm{MQ}&:= \sigma_\mathrm{MQ}/S_\mathrm{MQ}, \\
    R_\mathrm{AK}&:=\sigma_\mathrm{AK}/S_\mathrm{AK}.
\end{align*}

In [43]:
### ---- step 9: recompute and validate the source Gaussian widths ---------

def sigma_MQ(sigma_s, sigma_R, sigma_r, aM, bM):
    r"""sigma_MQ = 1/sqrt(2(1/sigma_s^2 + aM/sigma_R^2 + bM/sigma_r^2))."""
    assert PREC(sigma_s) > 0 and PREC(sigma_R) > 0 and PREC(sigma_r) > 0
    inv2 = 2*(1/PREC(sigma_s)^2 + PREC(aM)/PREC(sigma_R)^2
              + PREC(bM)/PREC(sigma_r)^2)
    return 1/inv2.sqrt()

def sigma_AK(sigma_s, sigma_R, sigma_r, bA, beta_Q):
    r"""sigma_AK = 1/(beta_Q sqrt(2(max{1/sigma_s^2, 1/sigma_R^2} + bA/sigma_r^2)))."""
    assert PREC(sigma_s) > 0 and PREC(sigma_R) > 0 and PREC(sigma_r) > 0
    assert PREC(beta_Q) > 0
    inv2 = 2*(max(1/PREC(sigma_s)^2, 1/PREC(sigma_R)^2)
              + PREC(bA)/PREC(sigma_r)^2)
    return 1/(PREC(beta_Q) * inv2.sqrt())

def R_MQ(sigma_s, sigma_R, sigma_r, aM, bM, kappa, d, ell):
    r"""R_MQ = sigma_MQ / S_MQ.  Feasibility requires R_MQ >= 1."""
    return sigma_MQ(sigma_s, sigma_R, sigma_r, aM, bM) / S_MQ(kappa, d, ell)

def R_AK(sigma_s, sigma_R, sigma_r, bA, beta_Q, kappa, k, d, ell):
    r"""R_AK = sigma_AK / S_AK.  Feasibility requires R_AK >= 1."""
    return (sigma_AK(sigma_s, sigma_R, sigma_r, bA, beta_Q)
            / S_AK(kappa, k, d, ell))

Computing the source MLWE widths at one grid point.

In [44]:
### ---- step 9: source MLWE widths at one grid point ----------------------

def sigma_MLWE_MQ_AK(key):
    r"""sigma_MQ, sigma_AK, R_MQ, R_AK at key = (n, kappa, ell).

    Reads the optimized widths from WIDTHS and the coefficients from COEFF;
    k and beta_Q come from the GEOMETRY entry with this n.
    """
    n, kappa, ell = key
    c, w = COEFF[key], WIDTHS[key]
    g = next(G for G in GEOMETRY if G['n'] == n)
    sig = (w['sigma_s'], w['sigma_R'], w['sigma_r'])
    return dict(
        sigma_MQ=sigma_MQ(*sig, c['a_MQ'], c['b_MQ']),
        sigma_AK=sigma_AK(*sig, c['b_AK'], g['beta_Q']),
        R_MQ=R_MQ(*sig, c['a_MQ'], c['b_MQ'], kappa, d, ell),
        R_AK=R_AK(*sig, c['b_AK'], g['beta_Q'], kappa, g['k'], d, ell),
    )

Constructing the table.

In [45]:
### ---- step 9: source MLWE widths over the search grid -------------------

SIGMA_MLWE = {key: sigma_MLWE_MQ_AK(key) for key in COEFF}
assert_prec(SIGMA_MLWE, 'SIGMA_MLWE', ('sigma_MQ', 'sigma_AK', 'R_MQ', 'R_AK'))

TOL = PREC(2)^(-100)
for key, s in SIGMA_MLWE.items():
    assert s['R_MQ'] >= 1 - TOL, f"{key}: R_MQ = {s['R_MQ']} < 1 (infeasible)"
    assert s['R_AK'] >= 1 - TOL, (
        f"{key}: R_AK = {s['R_AK']}. The AK constraint is active, so R_AK = 1 "
        f"exactly; a deficit above 2^-190 indicates lost precision upstream, "
        f"not infeasibility.")


def sigma_MLWE_table(L=None):
    r"""Print the source MLWE widths and smoothing ratios for ell in L."""
    L = list(LIST_ELL) if L is None else list(L)
    assert all(ell in LIST_ELL for ell in L), \
        f"not in LIST_ELL: {[ell for ell in L if ell not in LIST_ELL]}"
    print(f"{'n':>3} {'kap':>4} {'ell':>4} | "
          f"{'lg sigma_MQ':>12} {'lg sigma_AK':>12} "
          f"{'lg S_MQ':>9} {'lg S_AK':>9} | {'R_MQ':>8} {'R_AK':>8}")
    for g in GEOMETRY:
        for kappa in KAPPAS:
            for ell in L:
                key = (g['n'], kappa, ell)
                s, S = SIGMA_MLWE[key], SMOOTH[key]
                print(f"{g['n']:>3} {kappa:>4} {ell:>4} | "
                      f"{lg(s['sigma_MQ']):>12.4f} {lg(s['sigma_AK']):>12.4f} "
                      f"{lg(S['S_MQ']):>9.4f} {lg(S['S_AK']):>9.4f} | "
                      f"{s['R_MQ']:>8.4f} {s['R_AK']:>8.4f}")

Printing a table.

In [46]:
sigma_MLWE_table([8, 12, 16, 20, 24])

  n  kap  ell |  lg sigma_MQ  lg sigma_AK   lg S_MQ   lg S_AK |     R_MQ     R_AK
 40  128    8 |       5.1385       3.0113    2.9898    3.0113 |   4.4344   1.0000
 40  128   12 |       5.1414       3.0141    2.9927    3.0141 |   4.4341   1.0000
 40  128   16 |       5.1434       3.0162    2.9948    3.0162 |   4.4339   1.0000
 40  128   20 |       5.1450       3.0177    2.9964    3.0177 |   4.4338   1.0000
 40  128   24 |       5.1462       3.0190    2.9978    3.0190 |   4.4336   1.0000
 40  192    8 |       5.3988       3.2715    3.2566    3.2715 |   4.4142   1.0000
 40  192   12 |       5.4008       3.2735    3.2586    3.2735 |   4.4141   1.0000
 40  192   16 |       5.4022       3.2749    3.2601    3.2749 |   4.4140   1.0000
 40  192   20 |       5.4033       3.2760    3.2612    3.2760 |   4.4139   1.0000
 40  192   24 |       5.4042       3.2769    3.2621    3.2769 |   4.4139   1.0000
 40  256    8 |       5.5896       3.4624    3.4509    3.4624 |   4.4035   1.0000
 40  256   12 | 

# Computing the Response and Extraction Bounds

We compute:
1. $\beta_\mathbf{s} = \theta_\mathbf{s} \sigma_\mathbf{s}\sqrt{d\ell}$, where $\theta_\mathbf{s} \geq \min\left\{1+\frac{\kappa\ln 2}{3d\ell}, 1 + \sqrt{\frac{4\kappa\ln2}{d\ell}}\right\}$,
2. $\beta_\mathbf{R} = \theta_\mathbf{R}\sigma_\mathbf{R}$, where $\theta_\mathbf{R}\geq\min\left\{1+\frac{\kappa_\mathbf{R}\ln 2}{3},1+\sqrt{4\kappa_\mathbf{R}\ln 2}\right\}$ and $\kappa_\mathbf{R}:= \kappa+\left\lceil\log_2((k-1)d\ell)\right\rceil$,
3. $\beta_\mathbf{r} = \theta_\mathbf{r}\sigma_\mathbf{r}\sqrt{d\ell}$, where $\theta_{\bm{r}} \geq \min\left\{ 1+\frac{\kappa_{\bm{r}}\ln 2}{3d\ell}, 1+\sqrt{\frac{4\kappa_{\bm{r}}\ln 2}{d\ell}}\right\}$, $\kappa_{\bm{r}} := \kappa+\left\lceil\log_2(nQ_\mathsf{Sign})\right\rceil$, and $Q_\mathsf{Sign}$ is the maximum number of signing queries.
and also use the value $B_1$ obtained from the minimize_B1_from_dicts.

We define $B_2 := \omega_{\bm{\lambda}}\beta_{\lambda}B_1$, $B_\mathsf{STMSIS} := \sqrt{B_2^2 + \omega_c}$, and $B_\mathsf{MSIS} := 2B_\mathsf{STMSIS}$.

In [47]:
def B_2(omega_lambda, beta_lambda, B_1):
    r"""B_2 = omega_lambda * beta_lambda * B_1."""
    assert omega_lambda >= 1 and beta_lambda >= 1 and PREC(B_1) > 0
    return PREC(omega_lambda) * PREC(beta_lambda) * PREC(B_1)

def B_STMSIS(B_2_val, omega_c):
    r"""B_STMSIS = sqrt(B_2^2 + omega_c), using ||c||_2^2 = omega_c for
    c with ||c||_inf = 1 and ||c||_1 = omega_c."""
    assert PREC(B_2_val) > 0 and omega_c >= 1
    return (PREC(B_2_val)^2 + PREC(omega_c)).sqrt()

def B_MSIS(B_STMSIS_val):
    r"""B_MSIS = 2 B_STMSIS (difference of two forgeries)."""
    assert PREC(B_STMSIS_val) > 0
    return 2 * PREC(B_STMSIS_val)

Computing the norms at one grid points.

In [48]:
def norm_bounds(key):
    r"""beta_s, beta_R, beta_r, B_1, B_2, B_STMSIS, B_MSIS at (n, kappa, ell).

    B_1 is taken from WIDTHS (the C-coefficient form) and cross-checked against the beta form, which validates the C_* definitions.
    """
    n, kappa, ell = key
    w = WIDTHS[key]
    g = next(G for G in GEOMETRY if G['n'] == n)
    wc, Q_S = OMEGA_C[kappa], QUERY[kappa]['Q_Sign']

    b_s = beta_s(w['sigma_s'], kappa, d, ell)
    b_R = beta_R(w['sigma_R'], kappa, g['k'], d, ell)
    b_r = beta_r(w['sigma_r'], kappa, n, d, ell, Q_S)

    B1_beta = wc * PREC(d*ell).sqrt() * (b_s + g['c_exp']*b_R) + b_r
    assert abs(B1_beta/w['B_1'] - 1) < PREC(2)^(-150), \
        f"{key}: B_1 mismatch, beta form {B1_beta} vs C form {w['B_1']}"

    B2v  = B_2(g['omega_lambda'], g['beta_lambda'], w['B_1'])
    BSTv = B_STMSIS(B2v, wc)
    return dict(beta_s=b_s, beta_R=b_R, beta_r=b_r, B_1=w['B_1'], B_2=B2v, B_STMSIS=BSTv, B_MSIS=B_MSIS(BSTv))

Constructing the table.

In [49]:
### ---- step 10: response and extraction bounds over the search grid ------

NORM_BOUNDS = {key: norm_bounds(key) for key in COEFF}
assert_prec(NORM_BOUNDS, 'NORM_BOUNDS', ('beta_s', 'beta_R', 'beta_r', 'B_1', 'B_2','B_STMSIS', 'B_MSIS'))

for key, b in NORM_BOUNDS.items():
    assert b['B_STMSIS'] >= b['B_2'], f"{key}: B_STMSIS < B_2"
    assert b['B_MSIS'] == 2*b['B_STMSIS'], f"{key}: B_MSIS != 2 B_STMSIS"
    # beta_r dominates B_1: the flooding noise carries the sqrt(Q_Sign) factor.
    assert b['beta_r'] / b['B_1'] > PREC(1)/2, \
        f"{key}: beta_r/B_1 = {b['beta_r']/b['B_1']}, expected ~0.8"


def norm_bounds_table(L=None):
    r"""Print the response and extraction bounds for ell in L."""
    L = list(LIST_ELL) if L is None else list(L)
    assert all(ell in LIST_ELL for ell in L), \
        f"not in LIST_ELL: {[ell for ell in L if ell not in LIST_ELL]}"
    print(f"{'n':>3} {'kap':>4} {'ell':>4} | "
          f"{'lg beta_s':>10} {'lg beta_R':>10} {'lg beta_r':>10} | "
          f"{'lg B_1':>9} {'lg B_2':>9} {'lg B_MSIS':>10}")
    for g in GEOMETRY:
        for kappa in KAPPAS:
            for ell in L:
                b = NORM_BOUNDS[(g['n'], kappa, ell)]
                print(f"{g['n']:>3} {kappa:>4} {ell:>4} | "
                      f"{lg(b['beta_s']):>10.4f} {lg(b['beta_R']):>10.4f} "
                      f"{lg(b['beta_r']):>10.4f} | "
                      f"{lg(b['B_1']):>9.4f} {lg(b['B_2']):>9.4f} "
                      f"{lg(b['B_MSIS']):>10.4f}")

Printing a table.

In [50]:
norm_bounds_table([8, 12, 16, 20, 24])

  n  kap  ell |  lg beta_s  lg beta_R  lg beta_r |    lg B_1    lg B_2  lg B_MSIS
 40  128    8 |    22.7458    21.1312    52.3522 |   52.3522   59.3180    60.3180
 40  128   12 |    22.9849    21.0812    52.6423 |   52.6423   59.6081    60.6081
 40  128   16 |    23.1519    21.0472    52.8492 |   52.8492   59.8150    60.8150
 40  128   20 |    23.2807    21.0161    53.0101 |   53.0101   59.9759    60.9759
 40  128   24 |    23.3852    20.9897    53.1419 |   53.1419   60.1076    61.1076
 40  192    8 |    23.3021    21.9345    54.1703 |   54.1703   61.1361    62.1361
 40  192   12 |    23.5424    21.8875    54.4577 |   54.4577   61.4235    62.4235
 40  192   16 |    23.7109    21.8544    54.6631 |   54.6632   61.6289    62.6289
 40  192   20 |    23.8409    21.8250    54.8231 |   54.8231   61.7889    62.7889
 40  192   24 |    23.9463    21.8000    54.9541 |   54.9541   61.9199    62.9199
 40  256    8 |    23.4648    22.2776    54.8137 |   54.8137   61.7795    62.7795
 40  256   12 | 

# Screening the Candidate for Total Module Dimensions

## MLWE-based Screen

We define the preliminary modulus scale $q_{\min}(\ell) := 2^{\left\lfloor \lg B_\mathsf{MSIS}(\ell) \right\rfloor+1}$.

In [51]:
### ---- step 12a: preliminary modulus scale -------------------------------

def b_q_min(key):
    r"""Smallest exponent with 2^b_q > 2 B_MSIS(ell).

    The +2 rather than +1 is required by the estimator, which rejects a SIS instance with length_bound >= (q-1)/2 as trivially easy. 
    That is stricter than the mathematical condition B < q, under which q e_1 is the trivial kernel vector.
    """
    return floor(lg(NORM_BOUNDS[key]['B_MSIS'])) + 2


def q_min(key):
    r"""Representative modulus 2^b_q_min for the screen.  The modulus used in
    the scheme is a prime, selected in Step 15."""
    return 2^b_q_min(key)


Q_MIN = {key: q_min(key) for key in COEFF}

for key, q in Q_MIN.items():
    B = NORM_BOUNDS[key]['B_MSIS']
    assert (q - 1)/2 > B,  f"{key}: B_MSIS >= (q-1)/2, estimator rejects"
    assert (q/2 - 1)/2 <= B, f"{key}: b_q_min not minimal"

We consider the root Hermite factor $\delta_0(\beta) := \left(\frac{\beta}{2\pi e}(\pi\beta)^{1/\beta}\right)^{\frac{1}{2(\beta-1)}}$ attainable by BKZ with block size $\beta$ under the geometric series assumption and its heuristic relation $\lg\delta_0 \approx \frac{\lg^2(q/\sigma)}{4N\lg q}$.

In [52]:
### ---- step 12b: root-Hermite factor -------------------------------------

def delta_0(beta):
    r"""delta_0(beta) = ((beta/(2 pi e)) (pi beta)^(1/beta))^(1/(2(beta-1)))."""
    assert beta > 1
    b = PREC(beta)
    return ((b/(2*PI*PREC(1).exp())) * (PI*b)^(1/b))^(1/(2*(b-1)))

def lg_delta_0(beta):
    r"""lg delta_0(beta), as used in the screening relation."""
    return lg(delta_0(beta))

# Sanity: delta_0 is decreasing in beta, and delta_0(beta_kappa) sits in the range expected for these security levels.
assert all(delta_0(b) > delta_0(b+50) for b in (439, 658, 877))
assert PREC("1.0035") < delta_0(439) < PREC("1.0040"), delta_0(439)

With $N=\ell'd$, we define
\begin{align*}
    	\ell'_\mathrm{req}(\ell) := \max\left\{\frac{ \lg^2\!\left( q_{\min}(\ell)/ \sigma_{\mathrm{AK}}(\ell) \right) }{ 4d\, \lg q_{\min}(\ell)\, \lg\delta_0( \beta_{\mathsf{MLWE}} ) },\frac{ \lg^2\!\left( q_{\min}(\ell)/ \sigma_{\mathrm{MQ}}(\ell) \right) }{ 4d\, \lg q_{\min}(\ell)\, \lg\delta_0( \beta_\kappa ) } \right\}.
\end{align*}
Since $p\geq\ell'$ and $\ell=p+\ell'$, every admissible split satisfies $\ell' \leq \left\lfloor \frac{\ell}{2} \right\rfloor$. We therefore discard $\ell$ whenever $\left\lceil \ell'_\mathrm{req}(\ell) \right\rceil > \left\lfloor \frac{\ell}{2} \right\rfloor$. For feasible $\ell$, we denote $L_\ell := \max\left\{1,\left\lceil \ell'_\mathrm{req}(\ell)\right\rceil\right\}$ and $U_\ell := \left\lfloor \frac{\ell}{2} \right\rfloor$.

In [53]:
### ---- step 12c: the simplified MLWE screen ------------------------------

def ell_prime_req(key):
    r"""Minimum secret rank ell' for both MLWE instances to reach their
    targets at q = q_min(ell), under lg delta_0 ~ lg^2(q/sigma)/(4 N lg q)."""
    n, kappa, ell = key
    g = next(G for G in GEOMETRY if G['n'] == n)
    s = SIGMA_MLWE[key]
    lq = lg(Q_MIN[key])
    t_AK = lg(Q_MIN[key]/s['sigma_AK'])^2 / (4*d*lq*lg_delta_0(BETA_MLWE[(kappa, g['k'])]))
    t_MQ = lg(Q_MIN[key]/s['sigma_MQ'])^2 / (4*d*lq*lg_delta_0(BETA_KAPPA[kappa]))
    return max(t_AK, t_MQ)


ELL_REQ  = {key: ell_prime_req(key) for key in COEFF}
ELL_PRIME_MIN   = {key: max(1, ceil(ELL_REQ[key])) for key in COEFF}
U_SPLIT  = {key: floor(key[2]/2) for key in COEFF}        # p >= ell'
PASS_MLWE = {key: ELL_PRIME_MIN[key] <= U_SPLIT[key] for key in COEFF}

Consistency check.

In [54]:
for key in COEFF:
    assert ELL_REQ[key] > 0, f"{key}: ell_req = {ELL_REQ[key]}"
    assert U_SPLIT[key] == floor(key[2]/2)
    assert PASS_MLWE[key] == (ELL_PRIME_MIN[key] <= U_SPLIT[key])

First passing $\ell$ and table generator.

In [55]:
### ---- step 12d: screen results ------------------------------------------

def screen_MLWE_table(L=None, only_pass=False):
    r"""Print the simplified MLWE screen for ell in L.

    Columns L_ell and U_ell are the endpoints of the split enumeration:
        L_ell = ell'_min(ell) = max{1, ceil(ell'_req(ell))}   (MLWE screen)
        U_ell = floor(ell/2)                                  (p >= ell')
    In code these are ELL_PRIME_MIN and U_SPLIT.  The 'pass' column is the
    MLWE screen alone; the MSIS diagnostic of Step 12e tightens the upper
    endpoint further, so a 'yes' here need not survive PASS_BOTH.
    """
    L = list(LIST_ELL) if L is None else list(L)
    assert all(ell in LIST_ELL for ell in L), \
        f"not in LIST_ELL: {[ell for ell in L if ell not in LIST_ELL]}"
    print(f"{'n':>3} {'kap':>4} {'ell':>4} | {'lg q_min':>9} "
          f"{'ell_req':>9} | {'L_ell':>6} {'U_ell':>6} {'pass':>5}")
    for g in GEOMETRY:
        for kappa in KAPPAS:
            for ell in L:
                key = (g['n'], kappa, ell)
                if only_pass and not PASS_MLWE[key]:
                    continue
                print(f"{g['n']:>3} {kappa:>4} {ell:>4} | "
                      f"{lg(Q_MIN[key]):>9.0f} {ELL_REQ[key]:>9.4f} | "
                      f"{ELL_PRIME_MIN[key]:>6} {U_SPLIT[key]:>6} "
                      f"{('yes' if PASS_MLWE[key] else 'no'):>5}")


def first_passing_ell():
    r"""Smallest ell in LIST_ELL passing the MLWE screen, per (n, kappa).

    See first_passing_ell_both() in Step 12e for the combined verdict.
    """
    print(f"{'n':>3} {'kappa':>6} {'first ell':>10}")
    for g in GEOMETRY:
        for kappa in KAPPAS:
            hit = next((ell for ell in sorted(LIST_ELL)
                        if PASS_MLWE[(g['n'], kappa, ell)]), None)
            print(f"{g['n']:>3} {kappa:>6} {str(hit):>10}")

Printing a table.

In [56]:
first_passing_ell()
# screen_MLWE_table(range(8, 26))

  n  kappa  first ell
 40    128         12
 40    192         16
 40    256         18
 60    128         12
 60    192         16
 60    256         20


## MSIS-based Screen

For diagnostic purposes, we also apply a simplified $\mathsf{MSIS}$ screen. After coefficient embedding, the systematic matrix has $n_\mathsf{SIS}=pd$ rows and $m_\mathsf{SIS}=(\ell+1)d$ columns. For reduction dimension $D$, the GSA predicts the attainable Euclidean length $L_\mathsf{MSIS}(D) \approx \delta_0(\beta_\kappa)^D q^{pd/D}$. Set $D^*(\ell,p,q) := \sqrt{\frac{pd\lg q }{ \lg\delta_0(\beta_\kappa)}}$ and $D_\mathrm{eff}(\ell,p,q) := \min\left\{D^*(\ell,p,q), (\ell+1)d \right\}$. The resulting screening length is $L_\mathsf{MSIS}^{\mathrm{screen}}(\ell,p,q) := 2^{D_\mathrm{eff}\lg\delta_0(\beta_\kappa) + \frac{pd\lg q}{D_\mathrm{eff}}}$. When $D^*(\ell,p,q)\leq(\ell+1)d$, this simplifies to $L_{\mathsf{MSIS}}^{\mathrm{screen}}(\ell,p,q) = 2^{2\sqrt{ pd\lg q\lg\delta_0(\beta_\kappa)}}$. At $q=q_{\min}(\ell)$, the unrestricted-dimension expression gives the diagnostic threshold $p_\mathrm{req}(\ell) := \frac{\lg^2 B_{\mathsf{MSIS}}(\ell) }{4d\, \lg q_{\min}(\ell)\, \lg\delta_0(\beta_\kappa)}$. Thus, a split passes the simplified $\mathsf{MSIS}$ screen at the minimum modulus scale when $p \geq \left\lfloor p_\mathrm{req}(\ell) \right\rfloor+1$. We write $L_p := \max\{1, \left\lfloor p_\mathrm{req}(\ell) \right\rfloor+1\}$.

Recall that we have three conditions that constrain the split $\ell = p + \ell'$:
1. MLWE screen: $\ell \geq L_\ell := \max\{1,\lceil \ell'_\mathrm{red} \rceil\}$,
2. Certiafibility: $p \geq \ell'$, so the flattened MLWE is not underdeterminined. Since $p = \ell - \ell'$, we obtain $\ell' \leq \lfloor \ell /2 \rfloor$.
3. MSIS diagnostic: $p \geq L_p := \max\{1, \lfloor p_\mathrm{red} \rfloor + 1\}$. Since $p = \ell - \ell'$, we obtain $\ell' \leq \ell - L_p$.

From 1-3 above, we obtain $L_\ell \leq \ell' \leq \min\{U_\ell, \ell-L_p\}$. We write $\ell'_{\min}(\ell):=L_\ell$ and $\ell'_{\max}(\ell) = \min\{U_\ell,\ell-L_p\}$.

In [57]:
### ---- step 12e: the simplified MSIS diagnostic --------------------------

def p_req(key):
    r"""Diagnostic threshold on p at q = q_min(ell): the GSA-attainable length
    at the unrestricted optimal sub-dimension reaches B_MSIS iff p <= p_req,
    so security requires p > p_req."""
    n, kappa, ell = key
    return (lg(NORM_BOUNDS[key]['B_MSIS'])^2
            / (4*d*lg(Q_MIN[key])*lg_delta_0(BETA_KAPPA[kappa])))

def D_star(p, kappa, q):
    r"""Optimal reduction sub-dimension sqrt(p d lg q / lg delta_0(beta_kappa)).
    Not used in the enumeration; retained to flag where the unrestricted
    expression behind p_req is valid, i.e. p d <= D* <= (ell+1) d."""
    return (PREC(p*d)*lg(q) / lg_delta_0(BETA_KAPPA[kappa])).sqrt()


P_REQ  = {key: p_req(key) for key in COEFF}
L_P    = {key: max(1, floor(P_REQ[key]) + 1) for key in COEFF}   # p >= L_p
U_MSIS = {key: key[2] - L_P[key] for key in COEFF}               # ell' <= ell - L_p

# Both caps are upper bounds on ell', so the admissible range is
#     ell' in [ELL_PRIME_MIN, ELL_PRIME_MAX],
# with ELL_PRIME_MIN = L_ell from the MLWE screen (Step 12c) and
# ELL_PRIME_MAX = min{floor(ell/2), ell - L_p}: certifiability and MSIS.
ELL_PRIME_MAX = {key: min(U_SPLIT[key], U_MSIS[key]) for key in COEFF}
PASS_BOTH = {key: ELL_PRIME_MIN[key] <= ELL_PRIME_MAX[key] for key in COEFF}

for key in COEFF:
    ell = key[2]
    assert P_REQ[key] > 0 and L_P[key] >= 1, key
    assert ELL_PRIME_MAX[key] <= U_SPLIT[key], key    # never looser than p >= ell'
    for lp in range(ELL_PRIME_MIN[key], ELL_PRIME_MAX[key] + 1):
        assert ell - lp >= lp,       f"{key}: p < ell' at ell' = {lp}"
        assert ell - lp >= L_P[key], f"{key}: p < L_p at ell' = {lp}"

def screen_both_table(L=None, only_pass=False):
    r"""Print the combined MLWE + MSIS screen for ell in L.

    Columns:
        L_ell   = ell'_min(ell) = max{1, ceil(ell'_req(ell))}   (MLWE)
        U_ell   = floor(ell/2)                                  (p >= ell')
        L_p     = max{1, floor(p_req(ell)) + 1}                 (MSIS)
        ell'max = min{U_ell, ell - L_p}
    A split is admissible iff L_ell <= ell' <= ell'max.
    """
    L = list(LIST_ELL) if L is None else list(L)
    assert all(ell in LIST_ELL for ell in L), \
        f"not in LIST_ELL: {[ell for ell in L if ell not in LIST_ELL]}"
    print(f"{'n':>3} {'kap':>4} {'ell':>4} | {'ell_req':>9} {'p_req':>9} | "
          f"{'L_ell':>6} {'U_ell':>6} {'L_p':>5} {'ell_max':>8} | "
          f"{'range':>9} {'binds':>6}")
    for g in GEOMETRY:
        for kappa in KAPPAS:
            for ell in L:
                key = (g['n'], kappa, ell)
                if only_pass and not PASS_BOTH[key]:
                    continue
                lo, hi = ELL_PRIME_MIN[key], ELL_PRIME_MAX[key]
                rng = f"{lo}..{hi}" if lo <= hi else "empty"
                binds = ('MSIS' if U_MSIS[key] < U_SPLIT[key] else
                         'cert' if U_MSIS[key] > U_SPLIT[key] else 'tie')
                print(f"{g['n']:>3} {kappa:>4} {ell:>4} | "
                      f"{ELL_REQ[key]:>9.4f} {P_REQ[key]:>9.4f} | "
                      f"{lo:>6} {U_SPLIT[key]:>6} {L_P[key]:>5} {hi:>8} | "
                      f"{rng:>9} {binds:>6}")


def first_passing_ell_both():
    r"""Smallest ell in LIST_ELL with a non-empty admissible split range."""
    print(f"{'n':>3} {'kappa':>6} {'MLWE only':>10} {'MLWE+MSIS':>10}")
    for g in GEOMETRY:
        for kappa in KAPPAS:
            a = next((ell for ell in sorted(LIST_ELL)
                      if PASS_MLWE[(g['n'], kappa, ell)]), None)
            b = next((ell for ell in sorted(LIST_ELL)
                      if PASS_BOTH[(g['n'], kappa, ell)]), None)
            print(f"{g['n']:>3} {kappa:>6} {str(a):>10} {str(b):>10}")

Some testing.

In [58]:
first_passing_ell_both()
screen_both_table(range(8, 26))

  n  kappa  MLWE only  MLWE+MSIS
 40    128         12         12
 40    192         16         16
 40    256         18         19
 60    128         12         12
 60    192         16         16
 60    256         20         20
  n  kap  ell |   ell_req     p_req |  L_ell  U_ell   L_p  ell_max |     range  binds
 40  128    8 |    5.2614    5.3318 |      6      4     6        2 |     empty   MSIS
 40  128    9 |    5.2612    5.3467 |      6      4     6        3 |     empty   MSIS
 40  128   10 |    5.2611    5.3600 |      6      5     6        4 |     empty   MSIS
 40  128   11 |    5.2610    5.3721 |      6      5     6        5 |     empty    tie
 40  128   12 |    5.2609    5.3832 |      6      6     6        6 |      6..6    tie
 40  128   13 |    5.2608    5.3934 |      6      6     6        6 |      6..6   cert
 40  128   14 |    5.2607    5.4029 |      6      7     6        7 |      6..7   cert
 40  128   15 |    5.2606    5.4117 |      6      7     6        7 |      6..7   

# Enumerate the Admissible Module Splits

We enumerate the admissible pair of $(\ell,p)$ in ascending order.

In [59]:
### ---- step 13: enumerate the admissible module splits -------------------
# In (ell, p) coordinates, with ell' = ell - p, the three constraints are
#     ceil(ell/2) <= p                 certifiability (p >= ell'), Step 12c
#     p <= ell - ell'_min(ell)         MLWE screen, Step 12c
#     p >= L_p(ell)                    MSIS diagnostic, Step 12e
# The first two are enforced; the third is reported as a flag, since a larger
# modulus may rescue a split that fails it at q_min.

def split_list(n, kappa):
    r"""Admissible candidates for one (n, kappa) as (ell, p, ell', prioritized), ordered by (ell, p) ascending.

    ell first: every size metric is monotone in ell, so the smallest feasible ell dominates.  
    p second: smaller p means a smaller vk (t in R_q^p).  
    The p-order is not binding on the reported result, because Step 14 evaluates every admissible split at the retained ell before ranking.
    """
    out = []
    for ell in sorted(LIST_ELL):
        key = (n, kappa, ell)
        for p in range(ceil(ell/2), ell - ELL_PRIME_MIN[key] + 1):
            out.append((ell, p, ell - p, p >= L_P[key]))
    out.sort(key=lambda t: (t[0], t[1]))
    return out

SPLIT_LIST = {(g['n'], kappa): split_list(g['n'], kappa)
              for g in GEOMETRY for kappa in KAPPAS}

for (n, kappa), lst in SPLIT_LIST.items():
    for ell, p, lp, pri in lst:
        key = (n, kappa, ell)
        assert p + lp == ell,                    (n, kappa, ell, p, lp)
        assert p >= lp,                          f"{key}: p < ell' at p = {p}"
        assert lp >= ELL_PRIME_MIN[key],         f"{key}: ell' below MLWE screen"
        assert pri == (p >= L_P[key]),           f"{key}: wrong priority at p = {p}"
    assert len(lst) == len(set((e, p) for e, p, _, _ in lst)), (n, kappa)
    assert lst == sorted(lst, key=lambda t: (t[0], t[1])), (n, kappa)

Function for printing.

In [60]:
def split_list_table(n, kappa, first=None):
    r"""Print the split list for one (n, kappa); first=k shows only the head."""
    lst = SPLIT_LIST[(n, kappa)]
    npri = sum(1 for t in lst if t[3])
    print(f"n={n} kappa={kappa}: {len(lst)} candidates "
          f"({npri} passing the MSIS diagnostic at q_min)")
    print(f"  {'#':>3} {'ell':>4} {'p':>4} {'ell_prime':>10} {'MSIS ok':>8}")
    for i, (ell, p, lp, pri) in enumerate(lst if first is None else lst[:first]):
        print(f"  {i:>3} {ell:>4} {p:>4} {lp:>10} "
              f"{('yes' if pri else 'no'):>8}")

def split_list_summary():
    r"""Candidate counts and the first admissible ell, per (n, kappa)."""
    print(f"{'n':>3} {'kappa':>6} {'cands':>6} {'MSIS ok':>8} {'first ell':>10}")
    for g in GEOMETRY:
        for kappa in KAPPAS:
            lst = SPLIT_LIST[(g['n'], kappa)]
            npri = sum(1 for t in lst if t[3])
            first = lst[0][0] if lst else None
            print(f"{g['n']:>3} {kappa:>6} {len(lst):>6} {npri:>8} "
                  f"{str(first):>10}")

Summary results.

In [61]:
split_list_summary()

  n  kappa  cands  MSIS ok  first ell
 40    128     56       56         12
 40    192     30       30         16
 40    256     20       19         18
 60    128     56       56         12
 60    192     30       30         16
 60    256     12       12         20


# Search for the Modulus and Estimate the Lattice Problems

## Iteration Preparation

We search admissible prime moduli $q$ in increasing order, subject to $q>B_\mathsf{MSIS}(\ell)$, and, when NTT compatibility is required, $q\equiv1\pmod{2d}$. Writing $b_q:=\lfloor\lg q\rfloor+1$ for the bit length of $q$, we begin at $b_q^{\min}(\ell) := \left\lfloor \lg B_{\mathsf{MSIS}}(\ell) \right\rfloor+1 $ and initially search through $b_q \in \left\{b_q^{\min}(\ell),\dots, b_q^{\min}(\ell)+\Delta_q \right\}$, where $\Delta_q:=32$.

In [62]:
### ---- step 14a: modulus exponent sweep ----------------------------------
# b_q is the exponent, so the representative modulus is q = 2^b_q.  
# Prime selection is deferred: the first admissible prime above 2^b_q lies within ~2^-45 relative distance, which the estimator cannot resolve.
# b_q_min is defined in Step 12a, since Q_MIN = 2^b_q_min feeds the screen.

def b_q_range(key):
    r"""[b_q_min, b_q_min + DELTA_Q], ascending: MLWE weakens and MSIS
    strengthens with q, so the feasible set is an interval and the smallest
    admissible modulus is the best one."""
    lo = b_q_min(key)
    return range(lo, lo + DELTA_Q + 1)

for key in COEFF:
    assert 2^b_q_min(key) == Q_MIN[key], f"{key}: b_q_min inconsistent with Q_MIN"
    assert b_q_range(key)[0] == b_q_min(key)
    assert len(b_q_range(key)) == DELTA_Q + 1

Prime selection for later use.

In [63]:
def first_admissible_prime(key, b_q):
    r"""Smallest prime q > max(B_MSIS, 2^b_q), with q = 1 mod 2d if required."""
    lo = max(ceil(NORM_BOUNDS[key]['B_MSIS']), 2^b_q) + 1
    if not NTT_REQUIRED:
        return next_prime(lo - 1)
    m = 2*d
    q = lo + ((1 - lo) % m)
    while not is_prime(q):
        q += m
    return q

We downcast $\sigma_\mathrm{MQ}$, $\sigma_\mathrm{AK}$, and $B_\mathsf{MSIS}$ from PREC to float since ND.DiscreteGaussian and SIS.Parameters are NumPy-backed.

In [64]:
### ---- step 14b: cross the precision boundary ----------------------------
# The estimator is NumPy-backed, so real-valued inputs are float64 there.
# This is the one deliberate downcast in the pipeline: beta moves by ~6.4 per bit of lg(sigma), so a 2^-53 relative error cannot shift it.
# q is NOT downcast.  Powers of two survive float64, but an actual prime such as 2^60 + 30721 needs 60 mantissa bits and would round back to 2^60.

def estimator_inputs(key, b_q, q=None):
    r"""(q, sigma_MQ, sigma_AK, B_MSIS) ready for the estimator.

    q defaults to the representative 2^b_q; pass an exact prime to re-estimate
    a retained candidate.  q is returned as an exact Integer.
    """
    s, b = SIGMA_MLWE[key], NORM_BOUNDS[key]
    if q is None:
        q = 2^b_q
    assert q in ZZ, "q must stay an exact integer"
    return q, float(s['sigma_MQ']), float(s['sigma_AK']), float(b['B_MSIS'])

We want to estimate $\mathsf{MLWE}_{p,\ell',q,d,\sigma_\mathrm{MQ}(\ell)}$, $\mathsf{MLWE}_{p,\ell',q,d,\sigma_\mathrm{AK}(\ell)}$, and $\mathsf{MSIS}_{p,\ell'+1,q,d,B_\mathsf{MSIS}(\ell)}$. We consider the previous MLWE_to_LWE(params, n_rows, m_cols, q, d, sigma) with input $(p,\ell',q,d,\sigma)$ and MSIS_to_SIS_params(n_rows, m_random_cols, q, d, B_MSIS) with input $(p,\ell'+1,q,d,B_\mathsf{MSIS})$.

In [65]:
### ---- step 14c: build the three estimator parameter objects -------------

def build_params(key, b_q, p, ell_prime, q=None):
    r"""The three instances at (key, b_q, split), ready for the estimator.

        MLWE_{p, ell', q, d, sigma_MQ}   -> LWE_{ell' d, p d, q, sigma_MQ}
        MLWE_{p, ell', q, d, sigma_AK}   -> LWE_{ell' d, p d, q, sigma_AK}
        MSIS_{p, ell'+1, q, d, B_MSIS}   -> SIS_{p d, (ell+1) d, q, B_MSIS} ( where ell = p + ell')

    q defaults to the representative 2^b_q; pass an exact prime to re-estimate a retained candidate.  
    Real inputs are downcast to float here (Step 14b); q stays an exact Integer.
    """
    n, kappa, ell = key
    assert p + ell_prime == ell and p >= ell_prime, (key, p, ell_prime)
    q = 2^b_q if q is None else q
    assert q in ZZ and q > NORM_BOUNDS[key]['B_MSIS'], (key, b_q)

    sigma_MQ = float(SIGMA_MLWE[key]['sigma_MQ'])
    sigma_AK = float(SIGMA_MLWE[key]['sigma_AK'])
    B_MSIS   = float(NORM_BOUNDS[key]['B_MSIS'])
    stem = f"n={n},kap={kappa},ell={ell},p={p},lp={ell_prime},bq={b_q}"

    return dict(
        MQ   = MLWE_to_LWE_params(p, ell_prime, q, d, sigma_MQ, tag=f"MLWE-MQ {stem}"),
        AK   = MLWE_to_LWE_params(p, ell_prime, q, d, sigma_AK, tag=f"MLWE-AK {stem}"),
        MSIS = MSIS_to_SIS_params(p, ell_prime + 1, q, d, B_MSIS, norm=2, tag=f"MSIS {stem}"),
    )

Some testing.

In [66]:
key, p, lp = (40, 128, 12), 6, 6
b_q = b_q_min(key)                 # 61 after the shift, not 60
P = build_params(key, b_q, p, lp)
print(f"b_q = {b_q}")
print(P['MQ'])    # n = ell'*d = 3072 (secret dim), m = p*d = 3072 (samples)
print(P['AK'])    # n = ell'*d = 3072 (secret dim), m = p*d = 3072 (samples)
print(P['MSIS'])  # n = p*d = 3072 (rows),  m = (ell+1)*d = 6656 (columns)

b_q = 62
LWEParameters(n=3072, q=4611686018427387904, Xs=D(σ=35.29), Xe=D(σ=35.29), m=3072, tag='MLWE-MQ n=40,kap=128,ell=12,p=6,lp=6,bq=62')
LWEParameters(n=3072, q=4611686018427387904, Xs=D(σ=8.08), Xe=D(σ=8.08), m=3072, tag='MLWE-AK n=40,kap=128,ell=12,p=6,lp=6,bq=62')
SISParameters(n=3072, q=4611686018427387904, length_bound=1.7573064274531735e+18, m=6656, norm=2, tag='MSIS n=40,kap=128,ell=12,p=6,lp=6,bq=62')


Definining the function $\mathsf{Sec}$ such that $\mathsf{Sec}(\mathsf{MLWE}_{p,\ell',q,d,\sigma_\mathrm{MQ}(\ell)}) = \kappa_\mathrm{MQ}$, $\mathsf{Sec}(\mathsf{MLWE}_{p,\ell',q,d,\sigma_\mathrm{AK}(\ell)}) = \kappa_\mathrm{AK}$, and $\mathsf{Sec}(\mathsf{MSIS}_{p,\ell'+1,q,d,B_\mathsf{MSIS}(\ell)})=\kappa_\mathrm{M}$.

In [67]:
### ---- step 14d: Sec of the three instances ------------------------------

def Sec(params, kind, rough=False, cost_model=RC.ADPS16, verbose=False):
    r"""Concrete bit-security of one instance, under the given cost model.

    kind is 'MLWE' or 'MSIS'.  
    rough=True uses LWE.estimate.rough (usvp and dual_hybrid only, ADPS16 hard-coded), whose value upper-bounds the full estimate: valid for pruning, never for retention.  
    Ignored for MSIS, which costs a single attack either way.

    Returns lg of the cheapest finite rop, or None if no attack was costed.
    """
    assert kind in ('MLWE', 'MSIS'), kind
    if kind == 'MSIS':
        _, _, _, s = run_SIS_full_analysis(params, cost_model=cost_model,verbose=verbose)
    elif rough:
        _, _, _, s = run_LWE_rough_analysis(params, verbose=verbose)
    else:
        _, _, _, s = run_LWE_full_analysis(params, cost_model=cost_model,verbose=verbose)
    return s


def Sec_all(key, b_q, p, ell_prime, q=None, rough=False,cost_model=RC.ADPS16, verbose=False):
    r"""{'MQ': .., 'AK': .., 'MSIS': ..} at one candidate.

    AK is evaluated first: it carries the highest target (kappa_MLWE) against the smaller width, so it is the constraint most likely to fail.
    """
    P = build_params(key, b_q, p, ell_prime, q=q)
    return dict(
        AK   = Sec(P['AK'],   'MLWE', rough, cost_model, verbose),
        MQ   = Sec(P['MQ'],   'MLWE', rough, cost_model, verbose),
        MSIS = Sec(P['MSIS'], 'MSIS', False, cost_model, verbose),
    )

Retention test, we ignore the square-root introduced by the forking lemma, hence we set $\kappa_\mathsf{MSIS} = \kappa$. We also set the target $\kappa_\mathrm{MQ} = \kappa$ and $\kappa_\mathsf{MLWE}$ is obtained from the unforgeability analysis, $\kappa_\mathsf{MLWE}:= \lceil \kappa + 1 + \lg k\rceil$.

In [68]:
def targets(key):
    r"""{'MQ': kappa, 'AK': kappa_MLWE, 'MSIS': kappa_MSIS}.

    kappa_MSIS := kappa, ignoring the square-root loss of the forking lemma (Step 11); 
    kappa_MLWE := ceil(kappa + 1 + lg k) from the T_3 allocation.
    """
    n, kappa, _ = key
    g = next(G for G in GEOMETRY if G['n'] == n)
    return dict(MQ=kappa, AK=KAPPA_MLWE[(kappa, g['k'])], MSIS=kappa)


def is_retained(key, secs, strict_None=True):
    r"""True iff all three targets are met.

    A None Sec means no attack was costed, which indicates a degenerate instance rather than a hard one; it is treated as a failure.
    """
    t = targets(key)
    assert set(secs) == set(t), f"{key}: key mismatch {set(secs)} vs {set(t)}"
    if any(secs[j] is None for j in t):
        assert not strict_None, f"{key}: uncosted instance in {secs}"
        return False
    return all(secs[j] >= t[j] for j in t)

Some testing.

In [69]:
%%time
key, p, lp = (40, 128, 12), 6, 6
secs = Sec_all(key, b_q_min(key), p, lp)
t = targets(key)
for j in ('MQ', 'AK', 'MSIS'):
    print(f"{j:>4}: Sec = {secs[j]:7.1f}   target = {t[j]:>4}   "
          f"{'ok' if secs[j] >= t[j] else 'FAIL'}")
print(f"retained: {is_retained(key, secs)}")

usvp                 :: rop: ≈2^158.3, red: ≈2^158.3, δ: 1.003213, β: 542, d: 6118, tag: usvp
bdd                  :: rop: ≈2^158.7, red: ≈2^158.6, svp: ≈2^155.3, β: 543, η: 531, d: 6142, tag: bdd
dual                 :: rop: ≈2^159.7, mem: ≈2^122.6, m: ≈2^11.6, β: 547, d: 6144, ↻: 1, tag: dual
dual_hybrid          :: rop: ≈2^159.4, red: ≈2^159.4, guess: ≈2^128.7, β: 546, p: 3, ζ: 0, t: 0, β': 546, N: ≈2^112.4, m: ≈2^11.6
usvp                 :: rop: ≈2^175.8, red: ≈2^175.8, δ: 1.002979, β: 602, d: 6104, tag: usvp
bdd                  :: rop: ≈2^176.1, red: ≈2^175.8, svp: ≈2^173.7, β: 602, η: 594, d: 6140, tag: bdd
dual                 :: rop: ≈2^177.5, mem: ≈2^135.4, m: ≈2^11.6, β: 608, d: 6144, ↻: 1, tag: dual
dual_hybrid          :: rop: ≈2^177.0, red: ≈2^177.0, guess: ≈2^145.9, β: 606, p: 3, ζ: 0, t: 0, β': 606, N: ≈2^125.3, m: ≈2^11.6
lattice  :: rop: ≈2^149.5, red: ≈2^149.5, δ: 1.003347, β: 512, d: 6285, tag: euclidean
  MQ: Sec =   175.8   target =  128   ok
  AK: Sec =   158.3 

Some checking (the attacks from LWE analysis).

In [70]:
%%time
res, _, _, _ = run_LWE_full_analysis(P['AK'], verbose=False)
print(sorted(res.keys()))

usvp                 :: rop: ≈2^158.3, red: ≈2^158.3, δ: 1.003213, β: 542, d: 6118, tag: usvp
bdd                  :: rop: ≈2^158.7, red: ≈2^158.6, svp: ≈2^155.3, β: 543, η: 531, d: 6142, tag: bdd
dual                 :: rop: ≈2^159.7, mem: ≈2^122.6, m: ≈2^11.6, β: 547, d: 6144, ↻: 1, tag: dual
dual_hybrid          :: rop: ≈2^159.4, red: ≈2^159.4, guess: ≈2^128.7, β: 546, p: 3, ζ: 0, t: 0, β': 546, N: ≈2^112.4, m: ≈2^11.6
['bdd', 'bdd_hybrid', 'bdd_mitm_hybrid', 'dual', 'dual_hybrid', 'usvp']
CPU times: user 3.37 s, sys: 0 ns, total: 3.37 s
Wall time: 3.37 s


## The Iteration

Previously, from experiments, we have $\sigma_\mathrm{MQ} > \sigma_\mathrm{AK}$ for all of our candidates. This implies $\mathsf{Sec}(\mathsf{MLWE}_{p,\ell',q,d,\sigma_\mathrm{MQ}(\ell)}) \geq \mathsf{Sec}(\mathsf{MLWE}_{p,\ell',q,d,\sigma_\mathrm{AK}(\ell)})$. We also have $\kappa_\mathsf{MLWE} \geq \kappa$ by construction. Therefore:
\begin{align*}
    \left(\mathsf{Sec}(\mathsf{MLWE}_{p,\ell',q,d,\sigma_\mathrm{AK}(\ell)}) \geq \kappa_\mathsf{MLWE}\right) \Rightarrow \left(\mathsf{Sec}(\mathsf{MLWE}_{p,\ell',q,d,\sigma_\mathrm{MQ}(\ell)}) \geq \kappa\right) 
\end{align*}
This means, we only need to check $\mathsf{Sec}(\mathsf{MLWE}_{p,\ell',q,d,\sigma_\mathrm{AK}(\ell)}) \geq \kappa_\mathsf{MLWE}$ for the MLWE hardness.

In [71]:
### ---- step 14e: rough screen for one candidate --------------------------

def find_rough_b_q(key, p, ell_prime, verbose=False):
    r"""Smallest b_q at which (p, ell') passes the rough screen, or None.

    Only the AK instance is tested on the MLWE side: sigma_MQ > sigma_AK at every grid point (asserted after Step 9) and kappa <= kappa_MLWE, 
    so Sec_AK >= kappa_MLWE implies Sec_MQ >= kappa.

    Sec_AK is maximal at b_q_min and decreases in b_q; Sec_MSIS increases in b_q.  
    So one AK call at b_q_min rejects a hopeless split outright, and the feasible set in b_q is an interval whose lower end is set by MSIS.

    Returns dict(b_q, Sec_AK, Sec_MSIS) or None.
    """
    n, kappa, ell = key
    g = next(G for G in GEOMETRY if G['n'] == n)
    t_AK = KAPPA_MLWE[(kappa, g['k'])]
    b_lo = b_q_min(key)

    P = build_params(key, b_lo, p, ell_prime)
    s_AK = Sec(P['AK'], 'MLWE', rough=True, verbose=verbose)
    if s_AK is None or s_AK < t_AK:
        return None                       # fails at its best b_q: dead

    for b_q in b_q_range(key):            # ascending; MSIS increases in b_q
        P = build_params(key, b_q, p, ell_prime)
        s_MSIS = Sec(P['MSIS'], 'MSIS', verbose=verbose)
        if s_MSIS is not None and s_MSIS >= kappa:
            break
    else:
        return None                       # for/else: MSIS never clears

    if b_q > b_lo:                        # AK may have fallen meanwhile
        P = build_params(key, b_q, p, ell_prime)
        s_AK = Sec(P['AK'], 'MLWE', rough=True, verbose=verbose)
        if s_AK is None or s_AK < t_AK:
            return None                   # feasible interval is empty

    return dict(b_q=b_q, Sec_AK=s_AK, Sec_MSIS=s_MSIS)

Function for the sweep using rough analysis.

In [72]:
def sweep_rough(n, kappa, stop_at_first=True, verbose=False):
    r"""Walk SPLIT_LIST for one (n, kappa), rough-screening each candidate.

    SPLIT_LIST is ascending in (ell, p), and the signature depends only on ell while vk grows with p, 
    so the first success is the size-optimal candidate.  
    stop_at_first=False continues through every split at the retained ell, for Step 15 to rank.
    """
    out, ell_star = [], None
    for ell, p, lp, pri in SPLIT_LIST[(n, kappa)]:
        if ell_star is not None and ell > ell_star:
            break
        hit = find_rough_b_q((n, kappa, ell), p, lp, verbose=verbose)
        if hit is None:
            continue
        P = build_params((n, kappa, ell), hit['b_q'], p, lp)
        hit.update(ell=ell, p=p, ell_prime=lp, prioritized=pri,
                   Sec_MQ=Sec(P['MQ'], 'MLWE', rough=True, verbose=verbose))
        assert hit['Sec_MQ'] >= kappa, f"MQ dominance failed at {(n,kappa,ell)}"
        out.append(hit)
        if stop_at_first:
            return out
        ell_star = ell
    return out

Printing the candidates.

In [73]:
%%time
ROUGH = {(g['n'], kappa): sweep_rough(g['n'], kappa, stop_at_first=False) for g in GEOMETRY for kappa in KAPPAS}

for (n, kappa), hits in ROUGH.items():
    h = hits[0] if hits else None
    print(f"n={n} kappa={kappa}: " +
          (f"ell={h['ell']} p={h['p']} l'={h['ell_prime']} b_q={h['b_q']} | "
           f"MQ={h['Sec_MQ']:.1f} AK={h['Sec_AK']:.1f} MSIS={h['Sec_MSIS']:.1f}"
           if h else "no candidate"))

usvp                 :: rop: ≈2^158.3, red: ≈2^158.3, δ: 1.003213, β: 542, d: 6118, tag: usvp
dual_hybrid          :: rop: ≈2^159.4, red: ≈2^159.4, guess: ≈2^128.7, β: 546, p: 3, ζ: 0, t: 0, β': 546, N: ≈2^112.4, m: ≈2^11.6
lattice  :: rop: ≈2^149.5, red: ≈2^149.5, δ: 1.003347, β: 512, d: 6285, tag: euclidean
usvp                 :: rop: ≈2^175.8, red: ≈2^175.8, δ: 1.002979, β: 602, d: 6104, tag: usvp
dual_hybrid          :: rop: ≈2^177.0, red: ≈2^177.0, guess: ≈2^145.9, β: 606, p: 3, ζ: 0, t: 0, β': 606, N: ≈2^125.3, m: ≈2^11.6
usvp                 :: rop: ≈2^224.0, red: ≈2^224.0, δ: 1.002493, β: 767, d: 8179, tag: usvp
dual_hybrid          :: rop: ≈2^225.7, red: ≈2^225.7, guess: ≈2^173.2, β: 773, p: 3, ζ: 0, t: 0, β': 773, N: ≈2^156.3, m: ≈2^12.0
lattice  :: rop: ≈2^212.3, red: ≈2^212.3, δ: 1.002594, β: 727, d: 8371, tag: euclidean
usvp                 :: rop: ≈2^247.0, red: ≈2^247.0, δ: 1.002317, β: 846, d: 8161, tag: usvp
dual_hybrid          :: rop: ≈2^248.8, red: ≈2^248.8, guess:

Printing the candidate from ROUGH.

In [74]:
for (n, kappa), hits in ROUGH.items():
    print(f"n={n} kappa={kappa}: {len(hits)} candidate(s)")
    for h in hits:
        print(f"   ell={h['ell']} p={h['p']} l'={h['ell_prime']} b_q={h['b_q']} | "
              f"MQ={h['Sec_MQ']:.1f} AK={h['Sec_AK']:.1f} MSIS={h['Sec_MSIS']:.1f}")

n=40 kappa=128: 1 candidate(s)
   ell=12 p=6 l'=6 b_q=62 | MQ=175.8 AK=158.3 MSIS=149.5
n=40 kappa=192: 1 candidate(s)
   ell=16 p=8 l'=8 b_q=64 | MQ=247.0 AK=224.0 MSIS=212.3
n=40 kappa=256: 1 candidate(s)
   ell=20 p=10 l'=10 b_q=65 | MQ=323.8 AK=294.9 MSIS=282.4
n=60 kappa=128: 1 candidate(s)
   ell=12 p=6 l'=6 b_q=65 | MQ=170.2 AK=147.8 MSIS=142.8
n=60 kappa=192: 1 candidate(s)
   ell=16 p=8 l'=8 b_q=67 | MQ=239.7 AK=209.9 MSIS=202.9
n=60 kappa=256: 1 candidate(s)
   ell=20 p=10 l'=10 b_q=67 | MQ=322.1 AK=282.7 MSIS=265.1


# Validate and Rank the Surviving Candidates

Validate the rough analysis result.

In [75]:
### ---- step 15a: select the modulus and validate one candidate -----------

def validate_hardness(key, p, ell_prime, b_q, verbose=False):
    r"""Re-estimate one candidate at an exact prime, with the full battery.

    The rough screen of Step 14 costs only usvp and dual_hybrid; the full estimate adds bdd, bdd_hybrid, bdd_mitm_hybrid and dual, so Sec can only fall.  
    The modulus changes from the representative 2^b_q to the nearest admissible prime, which shifts lg q by about 2^-45 and cannot move any estimate.

    Parameters:
        key:
            (n, kappa, ell).
        p, ell_prime:
            The module split, p + ell_prime = ell.
        b_q:
            Modulus exponent from the rough screen.
        verbose:
            Print each estimator summary.
    Returns:
        dict with the split, the exact prime q, the three Sec values, their
        margins over the targets, and a retained flag.
    """
    q = first_admissible_prime(key, b_q)
    assert is_prime(q), q
    assert 2*NORM_BOUNDS[key]['B_MSIS'] < q, f"{key}: B_MSIS >= (q-1)/2"
    assert (not NTT_REQUIRED) or q % (2*d) == 1, f"{key}: q != 1 mod 2d"

    secs = Sec_all(key, b_q, p, ell_prime, q=q, verbose=verbose)
    t = targets(key)
    return dict(
        ell=key[2], p=p, ell_prime=ell_prime, b_q=b_q, q=q,
        Sec_MQ=secs['MQ'], Sec_AK=secs['AK'], Sec_MSIS=secs['MSIS'],
        margin={j: (None if secs[j] is None else secs[j] - t[j]) for j in t},
        retained=is_retained(key, secs, strict_None=False),
    )

Validate the grid, one $(n,\kappa)$ at a time.

In [76]:
### ---- step 15b: validate the grid, one (n, kappa) at a time -------------
# FINAL accumulates across cells.  Run this cell once; re-running it discards
# results already computed.

FINAL = {}

def show_row(n, kappa, r):
    r"""One line per validated candidate, with margins over the targets."""
    m = r['margin']
    fmt = lambda j: (f"{r['Sec_' + j]:.1f} ({m[j]:+.1f})"
                     if r['Sec_' + j] is not None else "None")
    print(f"n={n} kap={kappa}: ell={r['ell']} p={r['p']} l'={r['ell_prime']} "
          f"b_q={r['b_q']} | MQ={fmt('MQ')}  AK={fmt('AK')}  "
          f"MSIS={fmt('MSIS')}  {'RETAINED' if r['retained'] else 'FAILED'}")

def validate_key(n, kappa, verbose=False):
    r"""Validate every rough-surviving candidate for one (n, kappa).

    Writes FINAL[(n, kappa)], overwriting any previous entry for that pair, and prints one line per candidate.
    """
    rows = [validate_hardness((n, kappa, h['ell']), h['p'],
                              h['ell_prime'], h['b_q'], verbose=verbose)
            for h in ROUGH[(n, kappa)]]
    FINAL[(n, kappa)] = rows
    for r in rows:
        show_row(n, kappa, r)
    return rows

The final table.

In [77]:
def final_table():
    r"""All validated candidates, in profile/kappa order."""
    print(f"{'n':>3} {'kap':>4} {'ell':>4} {'p':>3} {'l_p':>4} {'b_q':>4} | "
          f"{'Sec_MQ':>8} {'Sec_AK':>8} {'Sec_MSIS':>9} | {'ok':>4}")
    for g in GEOMETRY:
        for kappa in KAPPAS:
            for r in FINAL.get((g['n'], kappa), []):
                print(f"{g['n']:>3} {kappa:>4} {r['ell']:>4} {r['p']:>3} "
                      f"{r['ell_prime']:>4} {r['b_q']:>4} | "
                      f"{r['Sec_MQ']:>8.1f} {r['Sec_AK']:>8.1f} "
                      f"{r['Sec_MSIS']:>9.1f} | "
                      f"{('yes' if r['retained'] else 'NO'):>4}")
    missing = [(g['n'], kappa) for g in GEOMETRY for kappa in KAPPAS
               if (g['n'], kappa) not in FINAL]
    if missing:
        print(f"\nnot yet validated: {missing}")

## Validating at $n = 40$ and $\kappa = 128$.

In [78]:
%%time
validate_key(40, 128)      # ~72 s; already cross-checked at b_q=61

usvp                 :: rop: ≈2^158.3, red: ≈2^158.3, δ: 1.003213, β: 542, d: 6118, tag: usvp
bdd                  :: rop: ≈2^158.7, red: ≈2^158.6, svp: ≈2^155.3, β: 543, η: 531, d: 6142, tag: bdd
dual                 :: rop: ≈2^159.7, mem: ≈2^122.6, m: ≈2^11.6, β: 547, d: 6144, ↻: 1, tag: dual
dual_hybrid          :: rop: ≈2^159.4, red: ≈2^159.4, guess: ≈2^128.7, β: 546, p: 3, ζ: 0, t: 0, β': 546, N: ≈2^112.4, m: ≈2^11.6
usvp                 :: rop: ≈2^175.8, red: ≈2^175.8, δ: 1.002979, β: 602, d: 6104, tag: usvp
bdd                  :: rop: ≈2^176.1, red: ≈2^175.8, svp: ≈2^173.7, β: 602, η: 594, d: 6140, tag: bdd
dual                 :: rop: ≈2^177.5, mem: ≈2^135.4, m: ≈2^11.6, β: 608, d: 6144, ↻: 1, tag: dual
dual_hybrid          :: rop: ≈2^177.0, red: ≈2^177.0, guess: ≈2^145.9, β: 606, p: 3, ζ: 0, t: 0, β': 606, N: ≈2^125.3, m: ≈2^11.6
lattice  :: rop: ≈2^149.5, red: ≈2^149.5, δ: 1.003347, β: 512, d: 6285, tag: euclidean
n=40 kap=128: ell=12 p=6 l'=6 b_q=62 | MQ=175.8 (+47.8)  AK=1

[{'ell': 12,
  'p': 6,
  'ell_prime': 6,
  'b_q': 62,
  'q': 4611686018427388039,
  'Sec_MQ': 175.784,
  'Sec_AK': 158.26399999999998,
  'Sec_MSIS': 149.504,
  'margin': {'MQ': 47.78399999999999,
   'AK': 24.26399999999998,
   'MSIS': 21.50399999999999},
  'retained': True}]

## Validating at $n = 40$ and $\kappa = 192$.

In [79]:
%%time
validate_key(40, 192)

usvp                 :: rop: ≈2^224.0, red: ≈2^224.0, δ: 1.002493, β: 767, d: 8179, tag: usvp
bdd                  :: rop: ≈2^224.8, red: ≈2^224.0, svp: ≈2^223.7, β: 767, η: 765, d: 8186, tag: bdd
dual                 :: rop: ≈2^226.0, mem: ≈2^170.2, m: ≈2^12.0, β: 774, d: 8192, ↻: 1, tag: dual
dual_hybrid          :: rop: ≈2^225.7, red: ≈2^225.7, guess: ≈2^173.2, β: 773, p: 3, ζ: 0, t: 0, β': 773, N: ≈2^156.3, m: ≈2^12.0
usvp                 :: rop: ≈2^247.0, red: ≈2^247.0, δ: 1.002317, β: 846, d: 8161, tag: usvp
bdd                  :: rop: ≈2^247.4, red: ≈2^247.3, svp: ≈2^243.5, β: 847, η: 833, d: 8191, tag: bdd
dual                 :: rop: ≈2^249.1, mem: ≈2^186.7, m: ≈2^12.0, β: 853, d: 8192, ↻: 1, tag: dual
dual_hybrid          :: rop: ≈2^248.8, red: ≈2^248.8, guess: ≈2^196.3, β: 852, p: 3, ζ: 0, t: 0, β': 852, N: ≈2^175.2, m: ≈2^12.0
lattice  :: rop: ≈2^212.3, red: ≈2^212.3, δ: 1.002594, β: 727, d: 8371, tag: euclidean
n=40 kap=192: ell=16 p=8 l'=8 b_q=64 | MQ=247.0 (+55.0)  AK=2

[{'ell': 16,
  'p': 8,
  'ell_prime': 8,
  'b_q': 64,
  'q': 18446744073709551629,
  'Sec_MQ': 247.03199999999998,
  'Sec_AK': 223.964,
  'Sec_MSIS': 212.284,
  'margin': {'MQ': 55.03199999999998,
   'AK': 25.964,
   'MSIS': 20.283999999999992},
  'retained': True}]

## Validating at $n = 40$ and $\kappa = 256$.

In [80]:
%%time
validate_key(40, 256)

usvp                 :: rop: ≈2^294.9, red: ≈2^294.9, δ: 1.002028, β: 1010, d: 10193, tag: usvp
bdd                  :: rop: ≈2^295.3, red: ≈2^295.2, svp: ≈2^291.7, β: 1011, η: 998, d: 10233, tag: bdd
dual                 :: rop: ≈2^297.3, mem: ≈2^221.2, m: ≈2^12.3, β: 1018, d: 10240, ↻: 1, tag: dual
dual_hybrid          :: rop: ≈2^296.4, red: ≈2^296.4, guess: ≈2^284.2, β: 1015, p: 3, ζ: 10, t: 0, β': 1015, N: ≈2^209.0, m: ≈2^12.3
usvp                 :: rop: ≈2^323.8, red: ≈2^323.8, δ: 1.001888, β: 1109, d: 10226, tag: usvp
bdd                  :: rop: ≈2^324.6, red: ≈2^323.8, svp: ≈2^323.2, β: 1109, η: 1106, d: 10237, tag: bdd
dual                 :: rop: ≈2^326.5, mem: ≈2^242.1, m: ≈2^12.3, β: 1118, d: 10240, ↻: 1, tag: dual
dual_hybrid          :: rop: ≈2^326.2, red: ≈2^326.2, guess: ≈2^252.3, β: 1117, p: 3, ζ: 0, t: 0, β': 1117, N: ≈2^230.8, m: ≈2^12.3
lattice  :: rop: ≈2^282.4, red: ≈2^282.4, δ: 1.002096, β: 967, d: 10493, tag: euclidean
n=40 kap=256: ell=20 p=10 l'=10 b_q=65 | M

[{'ell': 20,
  'p': 10,
  'ell_prime': 10,
  'b_q': 65,
  'q': 36893488147419103363,
  'Sec_MQ': 323.828,
  'Sec_AK': 294.91999999999996,
  'Sec_MSIS': 282.364,
  'margin': {'MQ': 67.82799999999997,
   'AK': 32.91999999999996,
   'MSIS': 26.363999999999976},
  'retained': True}]

## Validating at $n = 60$ and $\kappa = 128$.

In [81]:
%%time
validate_key(60, 128)

usvp                 :: rop: ≈2^147.8, red: ≈2^147.8, δ: 1.003375, β: 506, d: 6088, tag: usvp
bdd                  :: rop: ≈2^148.1, red: ≈2^148.0, svp: ≈2^144.0, β: 507, η: 492, d: 6136, tag: bdd
dual                 :: rop: ≈2^149.2, mem: ≈2^115.0, m: ≈2^11.6, β: 511, d: 6144, ↻: 1, tag: dual
dual_hybrid          :: rop: ≈2^148.9, red: ≈2^148.9, guess: ≈2^116.6, β: 510, p: 3, ζ: 0, t: 0, β': 510, N: ≈2^100.2, m: ≈2^11.6
usvp                 :: rop: ≈2^170.2, red: ≈2^170.2, δ: 1.003049, β: 583, d: 6123, tag: usvp
bdd                  :: rop: ≈2^170.8, red: ≈2^170.2, svp: ≈2^169.1, β: 583, η: 578, d: 6140, tag: bdd
dual                 :: rop: ≈2^172.0, mem: ≈2^131.4, m: ≈2^11.6, β: 589, d: 6144, ↻: 1, tag: dual
dual_hybrid          :: rop: ≈2^171.7, red: ≈2^171.7, guess: ≈2^139.1, β: 588, p: 3, ζ: 0, t: 0, β': 588, N: ≈2^116.8, m: ≈2^11.6
lattice  :: rop: ≈2^142.8, red: ≈2^142.8, δ: 1.003458, β: 489, d: 6328, tag: euclidean
n=60 kap=128: ell=12 p=6 l'=6 b_q=65 | MQ=170.2 (+42.2)  AK=1

[{'ell': 12,
  'p': 6,
  'ell_prime': 6,
  'b_q': 65,
  'q': 36893488147419103363,
  'Sec_MQ': 170.236,
  'Sec_AK': 147.75199999999998,
  'Sec_MSIS': 142.78799999999998,
  'margin': {'MQ': 42.23599999999999,
   'AK': 12.751999999999981,
   'MSIS': 14.787999999999982},
  'retained': True}]

## Validating at $n = 60$ and $\kappa = 192$.

In [82]:
%%time
validate_key(60, 192)

usvp                 :: rop: ≈2^209.9, red: ≈2^209.9, δ: 1.002615, β: 719, d: 8150, tag: usvp
bdd                  :: rop: ≈2^210.5, red: ≈2^209.9, svp: ≈2^208.8, β: 719, η: 714, d: 8191, tag: bdd
dual                 :: rop: ≈2^211.7, mem: ≈2^159.9, m: ≈2^12.0, β: 725, d: 8192, ↻: 1, tag: dual
dual_hybrid          :: rop: ≈2^211.4, red: ≈2^211.4, guess: ≈2^163.8, β: 724, p: 3, ζ: 0, t: 0, β': 724, N: ≈2^146.9, m: ≈2^12.0
usvp                 :: rop: ≈2^239.7, red: ≈2^239.7, δ: 1.002370, β: 821, d: 8180, tag: usvp
bdd                  :: rop: ≈2^240.5, red: ≈2^239.7, svp: ≈2^239.1, β: 821, η: 818, d: 8190, tag: bdd
dual                 :: rop: ≈2^241.8, mem: ≈2^181.5, m: ≈2^12.0, β: 828, d: 8192, ↻: 1, tag: dual
dual_hybrid          :: rop: ≈2^241.5, red: ≈2^241.5, guess: ≈2^193.5, β: 827, p: 3, ζ: 0, t: 0, β': 827, N: ≈2^170.6, m: ≈2^12.0
lattice  :: rop: ≈2^202.9, red: ≈2^202.9, δ: 1.002682, β: 695, d: 8428, tag: euclidean
n=60 kap=192: ell=16 p=8 l'=8 b_q=67 | MQ=239.7 (+47.7)  AK=2

[{'ell': 16,
  'p': 8,
  'ell_prime': 8,
  'b_q': 67,
  'q': 147573952589676412931,
  'Sec_MQ': 239.73199999999997,
  'Sec_AK': 209.94799999999998,
  'Sec_MSIS': 202.94,
  'margin': {'MQ': 47.73199999999997,
   'AK': 10.947999999999979,
   'MSIS': 10.939999999999998},
  'retained': True}]

## Validating at $n = 60$ and $\kappa = 256$.

In [83]:
%%time
validate_key(60, 256)

usvp                 :: rop: ≈2^282.7, red: ≈2^282.7, δ: 1.002094, β: 968, d: 10232, tag: usvp
bdd                  :: rop: ≈2^283.2, red: ≈2^282.9, svp: ≈2^280.3, β: 969, η: 959, d: 10236, tag: bdd
dual                 :: rop: ≈2^285.0, mem: ≈2^212.5, m: ≈2^12.3, β: 976, d: 10240, ↻: 1, tag: dual
dual_hybrid          :: rop: ≈2^284.1, red: ≈2^284.1, guess: ≈2^275.9, β: 973, p: 3, ζ: 10, t: 0, β': 973, N: ≈2^200.7, m: ≈2^12.3
usvp                 :: rop: ≈2^322.1, red: ≈2^322.1, δ: 1.001896, β: 1103, d: 10206, tag: usvp
bdd                  :: rop: ≈2^322.4, red: ≈2^322.1, svp: ≈2^320.0, β: 1103, η: 1095, d: 10239, tag: bdd
dual                 :: rop: ≈2^324.7, mem: ≈2^240.9, m: ≈2^12.3, β: 1112, d: 10240, ↻: 1, tag: dual
dual_hybrid          :: rop: ≈2^324.4, red: ≈2^324.4, guess: ≈2^248.4, β: 1111, p: 3, ζ: 0, t: 0, β': 1111, N: ≈2^225.2, m: ≈2^12.3
lattice  :: rop: ≈2^265.1, red: ≈2^265.1, δ: 1.002198, β: 908, d: 10407, tag: euclidean
n=60 kap=256: ell=20 p=10 l'=10 b_q=67 | MQ=322

[{'ell': 20,
  'p': 10,
  'ell_prime': 10,
  'b_q': 67,
  'q': 147573952589676412931,
  'Sec_MQ': 322.07599999999996,
  'Sec_AK': 282.656,
  'Sec_MSIS': 265.13599999999997,
  'margin': {'MQ': 66.07599999999996,
   'AK': 19.656000000000006,
   'MSIS': 9.135999999999967},
  'retained': True}]

## The Retained Parameters

In [84]:
final_table()

  n  kap  ell   p  l_p  b_q |   Sec_MQ   Sec_AK  Sec_MSIS |   ok
 40  128   12   6    6   62 |    175.8    158.3     149.5 |  yes
 40  192   16   8    8   64 |    247.0    224.0     212.3 |  yes
 40  256   20  10   10   65 |    323.8    294.9     282.4 |  yes
 60  128   12   6    6   65 |    170.2    147.8     142.8 |  yes
 60  192   16   8    8   67 |    239.7    209.9     202.9 |  yes
 60  256   20  10   10   67 |    322.1    282.7     265.1 |  yes


Computing $\lceil \lg q \rceil$, the number of bits needed to store an element of $\mathbb{Z}_q$.

In [85]:
def lg_q_ceil(q):
    r"""ceil(lg q): the number of bits needed to store an element of Z_q.

    Equals Integer(q).nbits() for every q that is not a power of two, since 2^(nbits-1) < q < 2^nbits gives nbits-1 < lg q < nbits.  
    Computed from the bit length rather than from log2(q): 
    for q just above a power of two, such as 2^61 + 15, float log2 returns exactly 61.0 and ceil gives 61 instead of 62.
    """
    q = Integer(q)
    assert q >= 2, q
    return q.nbits() - 1 if q.is_power_of(2) else q.nbits()

The modulus.

In [86]:
for (n, kappa), rows in FINAL.items():
    for r in rows:
        print(f"n={n} kap={kappa}: b_q={r['b_q']}  q={r['q']}  "
              f"ceil(lg q)={lg_q_ceil(r['q'])}")

n=40 kap=128: b_q=62  q=4611686018427388039  ceil(lg q)=63
n=40 kap=192: b_q=64  q=18446744073709551629  ceil(lg q)=65
n=40 kap=256: b_q=65  q=36893488147419103363  ceil(lg q)=66
n=60 kap=128: b_q=65  q=36893488147419103363  ceil(lg q)=66
n=60 kap=192: b_q=67  q=147573952589676412931  ceil(lg q)=68
n=60 kap=256: b_q=67  q=147573952589676412931  ceil(lg q)=68


Assembling the parameter.

In [87]:
### ---- step 15c: assemble the parameter table ----------------------------

def parameter_row(n, kappa, r):
    r"""One row of the LaTSEC parameter table, from a FINAL entry."""
    ell = r['ell']
    key = (n, kappa, ell)
    w, b = WIDTHS[key], NORM_BOUNDS[key]
    return dict(
        n=n, kappa=kappa,
        Q_H=QUERY[kappa]['Q_H'], Q_Sign=QUERY[kappa]['Q_Sign'],
        lg_q=lg_q_ceil(r['q']), q=r['q'],
        ell=ell, p=r['p'], ell_prime=r['ell_prime'],
        omega_c=OMEGA_C[kappa],
        lg_sigma_s=lg(w['sigma_s']), lg_sigma_R=lg(w['sigma_R']),
        lg_sigma_r=lg(w['sigma_r']),
        lg_B_1=lg(b['B_1']), lg_B_2=lg(b['B_2']),
        Sec_MQ=r['Sec_MQ'], Sec_AK=r['Sec_AK'], Sec_MSIS=r['Sec_MSIS'],
    )

Printing parameter table.

In [88]:
PARAMS = {(n, kappa): [parameter_row(n, kappa, r) for r in rows] for (n, kappa), rows in FINAL.items()}

def parameter_table():
    r"""Print the retained parameter sets in profile/kappa order."""
    head = (f"{'n':>3} {'kap':>4} {'lg Q_H':>7} {'lg Q_S':>7} {'lg q':>5} "
            f"{'p':>3} {'l_p':>4} {'w_c':>4} | "
            f"{'lg sig_s':>9} {'lg sig_R':>9} {'lg sig_r':>9} | "
            f"{'lg B_1':>8} {'lg B_2':>8}")
    print(head)
    for g in GEOMETRY:
        for kappa in KAPPAS:
            for r in PARAMS.get((g['n'], kappa), []):
                print(f"{r['n']:>3} {r['kappa']:>4} "
                      f"{lg(r['Q_H']):>7.0f} {lg(r['Q_Sign']):>7.0f} "
                      f"{r['lg_q']:>5} {r['p']:>3} {r['ell_prime']:>4} "
                      f"{r['omega_c']:>4} | "
                      f"{r['lg_sigma_s']:>9.4f} {r['lg_sigma_R']:>9.4f} "
                      f"{r['lg_sigma_r']:>9.4f} | "
                      f"{r['lg_B_1']:>8.4f} {r['lg_B_2']:>8.4f}")

Printing.

In [89]:
parameter_table()

  n  kap  lg Q_H  lg Q_S  lg q   p  l_p  w_c |  lg sig_s  lg sig_R  lg sig_r |   lg B_1   lg B_2
 40  128      63      62    63   6    6   30 |   16.6855   16.6855   46.3392 |  52.6423  59.6081
 40  192      64      64    65   8    8   44 |   17.2031   17.2031   48.1525 |  54.6632  61.6289
 40  256      64      64    66  10   10   60 |   17.3366   17.3366   48.7909 |  55.4624  62.4282
 60  128      63      62    66   6    6   30 |   17.2046   17.2046   47.1391 |  53.4422  62.1039
 60  192      64      64    68   8    8   44 |   17.7209   17.7209   48.9509 |  55.4615  64.1233
 60  256      64      64    68  10   10   60 |   17.8538   17.8538   49.5885 |  56.2600  64.9218


# Computing the Relevant Sizes of the Threshold Signature Scheme. 

## Computing the Partial and Aggregate Signature Sizes

For storage optimization, we consider ideal entropy encoding conditioned on the high-probability response bounds. Let $D:=d\ell$ be the coefficient dimension of a response vector in $\mathcal{R}^\ell$. The partial and aggregate responses satisfy $\|\bm{z}_i\|_2\leq B_1$ and $\|\bm{z}\|_2\leq B_2$, respectively, under the corresponding tail-good event. For a response vector with Euclidean bound $B$, we use the effective per-coordinate scale $B/\sqrt{D}$. The entropy of a centered Gaussian at this scale gives the ideal compressed-size estimate $\mathsf{Size}(B) := D\left(\lg\frac{B}{\sqrt{D}} + \frac{1}{2}\lg(2\pi e) \right) \approx D\left( \lg\frac{B}{\sqrt{D}} + 2.047 \right)$ bits. 
Equivalently, this is the leading-order logarithm of the volume of the $D$-dimensional Euclidean ball of radius $B$.

The fixed-weight challenge space satisfies $|\mathcal{C}| = 2^{\omega_c}\binom{d}{\omega_c}$, so we estimate the challenge encoding length by $b_c := \left\lceil \omega_c+\lg\binom{d}{\omega_c} \right\rceil$. Hence, for $\mathsf{sig}_i=(c,\bm{z}_i)$ and $\mathsf{sig}=(c,\bm{z})$, the high-probability ideal entropy-coded size estimates are $|\mathsf{sig}_i| \approx b_c+ d\ell\left( \lg\frac{B_1}{\sqrt{d\ell}} + 2.047 \right)$ and $|\mathsf{sig}|\approx b_c+ d\ell\left( \lg\frac{B_2}{\sqrt{d\ell}} + 2.047 \right)$.

These estimates hold under the corresponding tail-good event and exclude implementation-specific entropy-coding, framing, and byte-alignment overheads. 
If the common challenge $c$ is not retransmitted with each partial response, the per-signer communication is obtained by omitting $b_c$ from $|\mathsf{sig}_i|$.

In [90]:
### ---- step 16a: signature sizes -----------------------------------------
# Ideal entropy encoding conditioned on the high-probability response bounds.
# For a response vector in R^ell with Euclidean bound B and coefficient dimension D = d*ell, 
# the effective per-coordinate scale is B/sqrt(D) and Size(B) = D (lg(B/sqrt(D)) + (1/2) lg(2 pi e))   bits, which is lg Vol(ball_D(B)) to leading order.  
# Sizes hold under the corresponding tail-good event and exclude entropy-coder, framing, and byte-alignment overheads.

def b_c(kappa, d=d):
    r"""ceil(lg |C|) with |C| = 2^omega_c * binomial(d, omega_c)."""
    wc = OMEGA_C[kappa]
    return ceil(wc + lg(binomial(d, wc)))

def response_size(B, D):
    r"""Size(B) = D (lg(B/sqrt(D)) + (1/2) lg(2 pi e)) in bits."""
    assert PREC(B) > 0 and D >= 1
    return D * (lg(PREC(B)/PREC(D).sqrt()) + lg(2*PI*PREC(1).exp())/2)

def partial_sig_size(key, with_challenge=True):
    r"""|sig_i| in bits for sig_i = (c, z_i), with ||z_i||_2 <= B_1.

    with_challenge=False gives the per-signer communication when the common challenge is not retransmitted with each partial response.
    """
    n, kappa, ell = key
    size = response_size(NORM_BOUNDS[key]['B_1'], d*ell)
    return size + b_c(kappa) if with_challenge else size

def aggregate_sig_size(key):
    r"""|sig| in bits for sig = (c, z), with ||z||_2 <= B_2."""
    n, kappa, ell = key
    return b_c(kappa) + response_size(NORM_BOUNDS[key]['B_2'], d*ell)

Generating the table for the signature size.

In [91]:
def sig_size_table():
    print(f"{'n':>3} {'kap':>4} {'ell':>4} | "
          f"{'|sig_i| KB':>11} {'|sig| KB':>9}")
    for g in GEOMETRY:
        for kappa in KAPPAS:
            for r in FINAL.get((g['n'], kappa), []):
                key = (g['n'], kappa, r['ell'])
                kb = lambda x: float(x)/8/1024
                print(f"{g['n']:>3} {kappa:>4} {r['ell']:>4} | "
                      f"{kb(partial_sig_size(key)):>11.3f} "
                      f"{kb(aggregate_sig_size(key)):>9.3f}")

In [92]:
sig_size_table()

  n  kap  ell |  |sig_i| KB  |sig| KB
 40  128   12 |      36.321    41.545
 40  192   16 |      50.242    57.207
 40  256   20 |      63.600    72.307
 60  128   12 |      36.921    43.417
 60  192   16 |      51.040    59.702
 60  256   20 |      64.597    75.424


## Size of the Verification Key and Abort-Key

### Size of the Verification Key

The verification key is $(\bm{A},\bm{t})$ where $\bm{A} = \begin{bmatrix}\bm{I}_p \ \bm{A}'\end{bmatrix}$ and $\bm{A}' \stackrel{\$}{\gets}\mathcal{R}_q^{p \times (\ell-p)}$. The matrix $\bm{A}'$ is not stored in full, but is generated from the $256$-bit seed following the conventional seeded representation of public matrices in module-lattice schemes. For the public verification vector $\bm{t} \in \mathcal{R}_q^p$, it is stored in full precision without bit dropping compression, hence $\bm{t}$ requires $pd \lceil \lg q \rceil$ bits. Hence, we have $|\mathsf{vk}| = 256 + pd \lceil \lg q \rceil$.

In [93]:
### ---- step 16b: verification key size -----------------------------------
# vk = (A, t) with A = [I_p | A'] and A' expanded from a kappa-bit seed, so A' is not stored.  
# t in R_q^p is stored at full precision, without bit-dropping or rounding.

SEED_BITS = 256   # fixed-length seed for expanding A', following the conventional seeded representation of public matrices in module-lattice schemes (Kyber, Dilithium).

def size_vk(seed_bits, d, p, lg_q):
    r"""|vk| = seed_bits + p*d*ceil(lg q) bits."""
    assert seed_bits >= 1 and d >= 1 and p >= 1 and lg_q >= 1
    return seed_bits + p*d*lg_q

def size_vk_from_dict(key, r):
    r"""size_vk for a validated candidate r = FINAL[(n, kappa)][j] at key."""
    n, kappa, ell = key
    assert r['ell'] == ell, (key, r['ell'])
    return size_vk(SEED_BITS, d, r['p'], lg_q_ceil(r['q']))

Constructing the table.

In [94]:
def vk_size_table():
    print(f"{'n':>3} {'kap':>4} {'ell':>4} {'p':>3} {'lg q':>5} | "
          f"{'|vk| bits':>10} {'|vk| KB':>8}")
    for g in GEOMETRY:
        for kappa in KAPPAS:
            for r in FINAL.get((g['n'], kappa), []):
                key = (g['n'], kappa, r['ell'])
                bits = size_vk_from_dict(key, r)
                print(f"{g['n']:>3} {kappa:>4} {r['ell']:>4} {r['p']:>3} "
                      f"{lg_q_ceil(r['q']):>5} | {bits:>10} "
                      f"{float(bits)/8/1024:>8.3f}")

Printing the table.

In [95]:
vk_size_table()

  n  kap  ell   p  lg q |  |vk| bits  |vk| KB
 40  128   12   6    63 |     193792   23.656
 40  192   16   8    65 |     266496   32.531
 40  256   20  10    66 |     338176   41.281
 60  128   12   6    66 |     203008   24.781
 60  192   16   8    68 |     278784   34.031
 60  256   20  10    68 |     348416   42.531


### Size of the Abort-Key

The abort-key (per-party public verification material) $\mathsf{ak}_i$ is defined as $\mathsf{ak}_i = \bm{t}_i = \bm{A}\bm{s}_i \in \mathcal{R}_q^p$. We have $|\mathsf{ak}_i| = pd \lceil \lg q \rceil$. The abort-key for the scheme is defined as $\mathsf{ak} = (\mathsf{ak}_i)_{i \in [n]}$, and so $|\mathsf{ak}| = n \cdot |\mathsf{ak}_i| = npd \lceil \lg q  \rceil $.

In [96]:
### ---- step 16c: abort-key sizes -----------------------------------------
# ak_i = t_i = A s_i in R_q^p is per-party public verification material, stored at full precision without rounding, so |ak_i| = p d ceil(lg q).
# ak = (ak_i)_{i in [n]} must be public for non-interactive partial verification, hence |ak| = n |ak_i|.

def size_ak_i(d, p, lg_q):
    r"""|ak_i| = p*d*ceil(lg q) bits, independent of n and t_uf."""
    assert d >= 1 and p >= 1 and lg_q >= 1
    return p*d*lg_q

def size_ak_i_from_dict(key, r):
    r"""size_ak_i for a validated candidate r = FINAL[(n, kappa)][j] at key."""
    n, kappa, ell = key
    assert r['ell'] == ell, (key, r['ell'])
    return size_ak_i(d, r['p'], lg_q_ceil(r['q']))

def size_ak_total(n, d, p, lg_q):
    r"""|ak| = n * |ak_i| bits."""
    assert n >= 1
    return n * size_ak_i(d, p, lg_q)

def size_ak_total_from_dict(key, r):
    r"""size_ak_total for a validated candidate r = FINAL[(n, kappa)][j]."""
    n, kappa, ell = key
    assert r['ell'] == ell, (key, r['ell'])
    return size_ak_total(n, d, r['p'], lg_q_ceil(r['q']))

Constructing the table.

In [97]:
def ak_size_table():
    print(f"{'n':>3} {'kap':>4} {'ell':>4} {'p':>3} {'lg q':>5} | "
          f"{'|ak_i| MB':>10} {'|ak| MB':>9}")
    for g in GEOMETRY:
        for kappa in KAPPAS:
            for r in FINAL.get((g['n'], kappa), []):
                key = (g['n'], kappa, r['ell'])
                kb = lambda x: float(x)/8/(1024^2)
                print(f"{g['n']:>3} {kappa:>4} {r['ell']:>4} {r['p']:>3} "
                      f"{lg_q_ceil(r['q']):>5} | "
                      f"{kb(size_ak_i_from_dict(key, r)):>10.3f} "
                      f"{kb(size_ak_total_from_dict(key, r)):>9.3f}")

Printing the table.

In [98]:
ak_size_table()

  n  kap  ell   p  lg q |  |ak_i| MB   |ak| MB
 40  128   12   6    63 |      0.023     0.923
 40  192   16   8    65 |      0.032     1.270
 40  256   20  10    66 |      0.040     1.611
 60  128   12   6    66 |      0.024     1.450
 60  192   16   8    68 |      0.033     1.992
 60  256   20  10    68 |      0.042     2.490


## Communication Size

In the first signing round, we have $\mathsf{pm}_i^1 = \mathsf{cmt}_i \in \{0,1\}^{2\kappa}$. Thus, $|\mathsf{pm}_i^1| = 2\kappa$. In the second signing round, we have $\mathsf{pm}_i^2 = \bm{w}_i \in \mathcal{R}_q^p$. Thus, $|\mathsf{pm}_i^2| = pd \lceil \lg q \rceil$. In the third signing round, we have $\mathsf{pm}_i^3 = \bm{z}_i \in \mathcal{R}^\ell$. Previously, we have computed the size of $\bm{z}_i$ using the ideal entropy encoding conditioned on the high-probability response bounds, i.e., $|\bm{z}_i| \approx  d\ell \left(\lg \frac{B_1}{\sqrt{d\ell}} + 2.047\right)$.

Thus, if $\mathsf{Comm}_i$ denotes the per-party communication material, i.e., $\mathsf{Comm}_i$ denotes the message sent by party $i$ at round $1$, $2$, and $3$, we have $\mathsf{Comm}_i = 2\kappa + pd \lceil \lg q \rceil + |\bm{z}_i|$.

In [99]:
### ---- step 16d: per-party communication ---------------------------------
# Comm_i is what party i sends across the three signing rounds, and so is independent of |act|.  
# Round 1 sends a 2kappa-bit commitment binding w_i;
# round 2 opens it with w_i itself; 
# round 3 sends z_i without the challenge, which every party derives locally from the round-2 messages.

def size_comm_i(key, r):
    r"""(|pm^1|, |pm^2|, |pm^3|, |Comm_i|) in bits for a validated candidate."""
    n, kappa, ell = key
    assert r['ell'] == ell, (key, r['ell'])
    pm1 = 2*kappa
    pm2 = size_ak_i(d, r['p'], lg_q_ceil(r['q']))      # w_i in R_q^p
    pm3 = partial_sig_size(key, with_challenge=False)  # z_i in R^ell
    return pm1, pm2, pm3, pm1 + pm2 + pm3

Table for communication size.

In [100]:
def comm_table():
    kb = lambda x: float(x)/8/1024
    print(f"{'n':>3} {'kap':>4} | {'pm_1 b':>7} {'pm_2 KB':>8} {'pm_3 KB':>8} | "
          f"{'Comm_i KB':>10}")
    for g in GEOMETRY:
        for kappa in KAPPAS:
            for r in FINAL.get((g['n'], kappa), []):
                key = (g['n'], kappa, r['ell'])
                p1, p2, p3, tot = size_comm_i(key, r)
                print(f"{g['n']:>3} {kappa:>4} | {p1:>7} {kb(p2):>8.3f} "
                      f"{kb(p3):>8.3f} | {kb(tot):>10.3f}")

Printing the table.

In [101]:
comm_table()

  n  kap |  pm_1 b  pm_2 KB  pm_3 KB |  Comm_i KB
 40  128 |     256   23.625   36.298 |     59.954
 40  192 |     384   32.500   50.210 |     82.757
 40  256 |     512   41.250   63.561 |    104.873
 60  128 |     256   24.750   36.898 |     61.679
 60  192 |     384   34.000   51.009 |     85.056
 60  256 |     512   42.500   64.558 |    107.120


## Long-term Secret-Share Storage Per-Participant 

Each participant holds one long-term secret signing share $\mathsf{sk}_i := \bm{s}_i \in \mathcal{R}^{\ell}$. By $\mathbf{s}_i^\top = \mathbf{g}[i] \mathbf{s}^\top + \mathbf{\hat{G}}[i,\cdot]\mathbf{R}$, each coefficient of its coefficient embedding $\mathbf{s}_i\in\mathbb{Z}^{d\ell}$ satisfies $\mathbf{s}_i[j] = \mathbf{g}[i]\mathbf{s}[j] + \sum_{t=1}^{k-1} \mathbf{\hat{G}}[i,t]\mathbf{R}[t,j]$.  Since the summands are independent and $\|\mathbf{g}\|_\infty\leq\beta_\mathbf{g}=1$, the corresponding coefficient variance is bounded by $\sigma_{\mathbf{s}_i}^{2} = \mathbf{g}[i]^2\sigma_\mathbf{s}^2 + \sigma_\mathbf{R}^2 \sum_{t=1}^{k-1} \mathbf{\hat{G}}[i,t]^2 \leq \sigma_\mathbf{s}^2 + c_{\mathrm{exp}}\sigma_\mathbf{R}^2$. We therefore define the uniform effective coefficient scale $\sigma_{\mathsf{sk}_i} := \sqrt{\sigma_\mathbf{s}^2 + c_{\mathrm{exp}}\sigma_\mathbf{R}^2}$. Since, the selected instances satisfy $\sigma_\mathbf{s}=\sigma_\mathbf{R}$, this simplifies to $\sigma_{\mathsf{sk}_i} = \sigma_\mathbf{s}\sqrt{1+c_{\mathrm{exp}}}$. Hence, writing $D:=d\ell$, the ideal entropy-coded long-term secret-share size is estimated as $|\mathsf{sk}_i| \approx D\left( \lg\sigma_\mathbf{s} + \frac{1}{2}\lg(1+c_{\mathrm{exp}}) + \frac{1}{2}\lg(2\pi e) \right) \approx D\left( \lg\sigma_\mathbf{s} + \frac{1}{2}\lg(1+c_{\mathrm{exp}}) + 2.047 \right)$ bits.

In [102]:
### ---- step 16e: long-term secret-share storage --------------------------
# sk_i = s_i in R^ell, one share per participant.  
# Each coefficient has variance at most sigma_s^2 + c_exp sigma_R^2, so the uniform effective scale is sigma_sk_i = sqrt(sigma_s^2 + c_exp sigma_R^2); 
# with sigma_s = sigma_R this is sigma_s sqrt(1 + c_exp).

def size_sk_i(d, ell, sigma_s, c_exp):
    r"""|sk_i| = D (lg sigma_s + (1/2) lg(1 + c_exp) + (1/2) lg(2 pi e)) bits,
    with D = d*ell.  Assumes sigma_s = sigma_R; see size_sk_i_from_dict."""
    assert d >= 1 and ell >= 1 and PREC(sigma_s) > 0 and PREC(c_exp) > 0
    D = d*ell
    return D*(lg(sigma_s) + lg(1 + PREC(c_exp))/2 + lg(2*PI*PREC(1).exp())/2)

def size_sk_i_from_dict(key, r):
    r"""size_sk_i for a validated candidate r = FINAL[(n, kappa)][j] at key."""
    n, kappa, ell = key
    assert r['ell'] == ell, (key, r['ell'])
    g = next(G for G in GEOMETRY if G['n'] == n)
    w = WIDTHS[key]
    assert w['sigma_s'] == w['sigma_R'], \
        f"{key}: sigma_s != sigma_R, the simplified scale does not apply"
    return size_sk_i(d, ell, w['sigma_s'], g['c_exp'])

## Table Resume for All Relevant Sizes

Creating a table that summarizes relevant sizes quantities: for each profile $(n,\kappa)$, we provide $|\mathsf{vk}|$, $|\mathsf{ak}_i|$, $|\mathsf{ak}|$, $|\mathsf{sk}_i|$, $|\mathsf{sig}_i|$, $|\mathsf{sig}|$, and $\mathsf{Comm}_i$. All sizes are in KB. We consider the convention $1 \ KB = 1024 \ B$ and $1 \ MB = 1024 \ KB$.

In [103]:
### ---- step 16f: summary of relevant sizes -------------------------------

def relevant_size(key, r):
    r"""All reported sizes in bits for a validated candidate r at key.

    Returns dict with |sk_i|, |vk|, |ak_i|, |ak|, |sig_i|, |sig|, Comm_i.
    |sig_i| and |sig| include the challenge; Comm_i does not, since every
    party derives c locally from the round-2 messages.
    """
    n, kappa, ell = key
    assert r['ell'] == ell, (key, r['ell'])
    return dict(
        sk_i    = size_sk_i_from_dict(key, r),
        vk      = size_vk_from_dict(key, r),
        ak_i    = size_ak_i_from_dict(key, r),
        ak      = size_ak_total_from_dict(key, r),
        sig_i   = partial_sig_size(key),
        sig     = aggregate_sig_size(key),
        Comm_i  = size_comm_i(key, r)[3],
    )

Table of relevant size.

In [104]:
# Binary units throughout: 1 KB = 1024 B, 1 MB = 1024 KB.

def table_relevant_size():
    r"""Summary of reported sizes, all in KB (1 KB = 1024 B)."""
    KB = lambda x: float(x)/8/1024
    print(f"{'n':>3} {'kap':>4} {'ell':>4} {'p':>3} {'lg q':>5} | "
          f"{'sk_i':>9} {'vk':>9} {'ak_i':>9} {'ak':>10} | "
          f"{'sig_i':>9} {'sig':>9} {'Comm_i':>9}   (KB)")
    for g in GEOMETRY:
        for kappa in KAPPAS:
            for r in FINAL.get((g['n'], kappa), []):
                key = (g['n'], kappa, r['ell'])
                s = relevant_size(key, r)
                print(f"{g['n']:>3} {kappa:>4} {r['ell']:>4} {r['p']:>3} "
                      f"{lg_q_ceil(r['q']):>5} | "
                      f"{KB(s['sk_i']):>9.3f} {KB(s['vk']):>9.3f} "
                      f"{KB(s['ak_i']):>9.3f} {KB(s['ak']):>10.3f} | "
                      f"{KB(s['sig_i']):>9.3f} {KB(s['sig']):>9.3f} "
                      f"{KB(s['Comm_i']):>9.3f}")

Printing the table.

In [105]:
table_relevant_size()

  n  kap  ell   p  lg q |      sk_i        vk      ak_i         ak |     sig_i       sig    Comm_i   (KB)
 40  128   12   6    63 |    14.799    23.656    23.625    945.000 |    36.321    41.545    59.954
 40  192   16   8    65 |    20.250    32.531    32.500   1300.000 |    50.242    57.207    82.757
 40  256   20  10    66 |    25.480    41.281    41.250   1650.000 |    63.600    72.307   104.873
 60  128   12   6    66 |    15.189    24.781    24.750   1485.000 |    36.921    43.417    61.679
 60  192   16   8    68 |    20.768    34.031    34.000   2040.000 |    51.040    59.702    85.056
 60  256   20  10    68 |    26.126    42.531    42.500   2550.000 |    64.597    75.424   107.120
